# 이화여대 학사 자료 RAG 통합 노트북

**실행 순서:** 학사 선별 → 메뉴·오분류 검수 → 문서 재조립 → 적용 조건·날짜·관련 조문 구조화 → 결함 자료 보류 → JSONL → 선택적 Qdrant 적재.

이 파일 하나에서 처리합니다. 기존 분리 노트북은 백업 ZIP에만 보관합니다.

- 입력: 기존 `03_정제문서/documents_unique.jsonl`, `02_원본/`, 수집 대장. 원본 파일은 수정하지 않습니다.
- 결과: `10_학사선별/학부_학사_통합_v4_전달_sample/`. 기본 **문서 25개**, 확인 후 `HANDOFF_MODE="full"`로 확대합니다.
- 학술지 투고·간행 규정과 학부 학사 교육과정을 구분합니다. 짧은 실제 신청 단계도 보존합니다.
- 정상 파일: `documents.jsonl`, `chunks.jsonl`. 불완전한 본문과 미확정 FAQ 연결은 `quarantined_*.jsonl`로 보류합니다.
- `unknown`·`null`은 허용합니다. 제한 없음은 원문 근거가 있을 때만 `all`로 표시합니다.
- 표·예외·부칙·경과조치·과거 연도 문구를 보존합니다. 기존 PDF/HWP 파싱의 표 의미 검토 표시는 유지합니다.
- `RUN_QDRANT_LOAD=False`가 기본입니다. 실제 모델 임베딩·DB 적재는 True로 실행해야 합니다.

Python 3.10+에서 **커널 재시작 → 모든 셀 실행** 순서로 사용하세요. 마지막 검증 셀의 결과와 DB 상태는 별도로 확인합니다.

## 준비 및 실행 설정

선별·검수·JSONL 작성에는 현재 커널의 `IPython`, `lxml`이 필요합니다. 필요한 경우 별도 셀에서 `%pip install lxml`을 실행하세요.

DB 적재를 사용할 때는 `%pip install FlagEmbedding qdrant-client huggingface-hub`도 필요합니다. 첫 실행에서 BGE-M3 모델을 다운로드할 수 있으며 기본 임베딩 장치는 CPU입니다.

- `HANDOFF_MODE="sample"`: 20~30개 문서의 인계 형식을 먼저 확인 (기본 25).
- `HANDOFF_MODE="full"`: 전체 통과 문서의 인계 파일 생성.
- `RUN_QDRANT_LOAD=True`: 실제 임베딩과 Qdrant 적재. local은 PC 내부 DB, server는 `QDRANT_URL`/`QDRANT_API_KEY` 환경변수의 서버입니다.
- 비밀키를 노트북 코드/출력/JSONL/Git에 직접 넣지 마세요. server 접속 주소와 비밀키는 별도로 전달합니다.

공식 참고: [BGE-M3 모델·사용법](https://huggingface.co/BAAI/bge-m3), [Qdrant Python 클라이언트](https://github.com/qdrant/qdrant-client), [Qdrant 하이브리드 조회](https://qdrant.tech/documentation/search/hybrid-queries/).


## 1. 설정
`PROJECT_ROOT`를 직접 지정할 수 있습니다. 날짜는 기준일이며 게시일·시행일과 구분합니다. 실제 검토한 규정만 `CONFIRMED_CURRENT_IDS`에 넣으세요.

In [1]:
import csv, hashlib, heapq, html, json, os, random, re, unicodedata
from collections import Counter, defaultdict
from datetime import date, datetime
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit
from IPython.display import HTML, display

PROJECT_ROOT = None  # 예: Path("/내/폴더/이화여대_학사행정_RAG")
AS_OF_DATE = date(2026, 10, 6)
SAMPLE_SIZE = 200
SEED = 42
CHUNK_MAX_CHARS = 1800
CHECKPOINT_EVERY = 250
RUN_MODE = "full"
RUN_NAME = "학부_학사_통합_v4"
RUN_QUALITY_AUDIT = True
HANDOFF_MODE = "full"           # "sample" 또는 "full"
HANDOFF_SAMPLE_DOCUMENTS = 25     # 표본 문서 수: 20~30
RUN_QDRANT_LOAD = False           # True이면 모델 다운로드/임베딩/DB 적재 실행
QDRANT_MODE = "local"             # "local" 또는 "server"
QDRANT_COLLECTION_PREFIX = "ewha_undergraduate_v4"
QDRANT_DENSE_NAME = "bge_m3_dense"
QDRANT_USE_SPARSE = True
QDRANT_SPARSE_NAME = "bge_m3_sparse"
EMBEDDING_MODEL = "BAAI/bge-m3"
EMBEDDING_REVISION = "main"        # 적재 시 실제 커밋을 해석·고정·기록
EMBEDDING_DEVICE = "cpu"          # 필요하면 "cuda:0" 등 환경에 맞게 변경
EMBEDDING_BATCH_SIZE = 4
EMBEDDING_MAX_TOKENS = 8192
REPARSE_SELECTED_HTML = False  # 기존 추출이 의심스러운 본문만 로컬에서 재추출
CONFIRMED_CURRENT_IDS = set()  # 사람이 시행 상태를 확인한 원본 doc_id
CONFIRMED_SCOPE = {}          # {doc_id: "undergraduate" 또는 "graduate" 또는 "mixed"}

TOPIC_PATTERNS = {
    "학적": r"휴학|복학|재입학|학적|자퇴|제적|전과|학사경고|재학연한",
    "수강·학점·성적": r"수강|학점|성적|계절학기|강의평가|출석|교과목",
    "졸업·교육과정": r"졸업|수료|교육과정|교과과정|복수전공|부전공|전공.?이수|인증제",
    "등록·장학": r"등록금|등록.?기간|등록.?절차|등록.?안내|장학(?!재단)|학자금|분납",
    "학사일정": r"학사일정|개강|종강|수강신청.?기간|학위수여",
    "증명·신청": r"증명서|성적증명|재학증명|학생증|학적부|학적.?정정",
    "학칙·규정": r"학칙|학칙시행세칙|학사.?규정|학점.?인정.?규정|졸업.?요건",
}
HARD_NOISE = r"교수채용|교원채용|직원채용|강사.{0,8}채용|채용.{0,8}공고|방역|식단|예초|입찰|공사.?안내|사진첩|갤러리"
SOFT_NOISE = r"취업|채용|진로|세미나|특강|워크숍|공모전|홍보|학술대회|기숙사|동창|기부"
ACADEMIC_EXCEPTION = r"학점.{0,12}인정|졸업.{0,12}요건|자격시험.{0,12}대체|필수.{0,10}(이수|교과)|교과.{0,10}이수"
GRAD_HOSTS = {"graduate", "gsis", "gsti", "gsb", "lawschool", "ged", "gsd", "gsw", "thewha", "gspsewha", "performarts", "gsccph", "dentist", "tfl", "gsds"}
RULES_VERSION = "3.1"
HOUSING_PATTERN = r"기숙사|생활관|연합기숙|입주자|입사생|기숙사비"
HOUSING_ACTION_PATTERN = r"입주|입사|퇴사|퇴실|입실|모집|신청|선발|배정|연장|증명서"
HOUSING_ACADEMIC_PATTERN = r"휴학|복학|수강|학점.{0,12}인정|졸업.{0,12}(요건|기준)|학칙|장학금|장학생|학자금|등록금"
HOUSING_RX = re.compile(HOUSING_PATTERN,re.I)
HOUSING_ACTION_RX = re.compile(HOUSING_ACTION_PATTERN,re.I)
HOUSING_ACADEMIC_RX = re.compile(HOUSING_ACADEMIC_PATTERN,re.I)
TOPICS = {k: re.compile(v, re.I) for k,v in TOPIC_PATTERNS.items()}
HARD_RX, SOFT_RX = re.compile(HARD_NOISE,re.I), re.compile(SOFT_NOISE,re.I)
EXCEPTION_RX = re.compile(ACADEMIC_EXCEPTION,re.I)

if PROJECT_ROOT is None:
    candidates = []
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidates.extend([parent, parent/"이화여대_학사행정_RAG", parent/"빅데이터핀테크응용사례특강"/"이화여대_학사행정_RAG"])
    PROJECT_ROOT = next((p for p in candidates if (p/"03_정제문서/documents_unique.jsonl").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("설정 셀의 PROJECT_ROOT를 데이터 프로젝트 경로로 지정하세요.")
PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
INPUT = PROJECT_ROOT/"03_정제문서/documents_unique.jsonl"
OUTPUT_BASE = PROJECT_ROOT/"10_학사선별"
print("입력:", INPUT)
print("범위: 이화여대 전체 학부 / 기준일:", AS_OF_DATE)
if not 1 <= EMBEDDING_MAX_TOKENS <= 8192 or EMBEDDING_BATCH_SIZE < 1:
    raise ValueError("임베딩 토큰/배치 설정 범위 오류")
if RUN_QUALITY_AUDIT:
    try:
        from lxml import html as _required_lxml
    except ModuleNotFoundError as error:
        raise ModuleNotFoundError("품질 검수에는 lxml이 필요합니다. 현재 노트북 커널에서 %pip install lxml을 실행하세요.") from error


입력: /Users/gohanjun/Desktop/SNU/빅데이터핀테크응용사례특강/이화여대_학사행정_RAG/03_정제문서/documents_unique.jsonl
범위: 이화여대 전체 학부 / 기준일: 2026-10-06


## 2. 데이터 탐색과 출처 연결
한 줄씩 읽어 집계하며 본문 전체를 메모리에 쌓지 않습니다. 부모 문서 조회에는 제목·대상 단서만 저장합니다. URL 쿼리는 게시글 구분에 필요하므로 원본 그대로 보존합니다.

In [2]:
def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as f:
        for line_number, line in enumerate(f,1):
            if line.strip():
                try: yield json.loads(line)
                except json.JSONDecodeError as e:
                    raise ValueError(f"{path.name} {line_number}행 JSON 오류") from e

def source_urls(d):
    return list(dict.fromkeys([x for x in [d.get("source_url"),d.get("final_url"), *[v.get("source_url") for v in d.get("source_variants",[]) if isinstance(v,dict)]] if x]))

def compact(d):
    return {"doc_id":d.get("doc_id"), "title":d.get("title", ""), "source_url":d.get("source_url", ""), "parent_url":d.get("parent_url"), "text_hint":d.get("text", "")[:1400], "metadata":d.get("metadata",{}), "source_variants":d.get("source_variants",[])}

INDEX, PROFILE = {}, {k:Counter() for k in ["format","type","degree"]}
N_DOCUMENTS = 0
for d in read_jsonl(INPUT):
    record = compact(d)
    for url in source_urls(d): INDEX[url] = record
    PROFILE["format"][d.get("format","unknown")] += 1
    PROFILE["type"][d.get("metadata",{}).get("document_type","unknown")] += 1
    PROFILE["degree"][d.get("metadata",{}).get("degree_level","unknown")] += 1
    N_DOCUMENTS += 1
print(f"문서 {N_DOCUMENTS:,}건, 출처 URL {len(INDEX):,}개")

ORGANIZATIONS = []
org_path = PROJECT_ROOT/"01_출처목록/조직별_공식목록_대조.jsonl"
if org_path.exists():
    for o in read_jsonl(org_path):
        u = urlsplit(o.get("homepage_url") or "")
        if u.hostname:
            ORGANIZATIONS.append((u.hostname.lower(),u.path.rstrip("/") or "/",o.get("organization_name","unknown")))
ORGANIZATIONS.sort(key=lambda row:len(row[1]),reverse=True)
FAQ_BY_DOC = defaultdict(list)
faq_path = PROJECT_ROOT/"04_구조화정보/FAQ_질문답변_쌍.jsonl"
if faq_path.exists():
    for q in read_jsonl(faq_path): FAQ_BY_DOC[q.get("source_doc_id")].append(q)

# FAQ가 다른 대표 URL로 중복 통합된 경우도 출처를 대조한다.
FAQ_BY_URL = defaultdict(list)
for qs in list(FAQ_BY_DOC.values()):
    for q in qs:
        for url in [q.get("source_url"),*q.get("source_variants",[])]:
            if isinstance(url,str): FAQ_BY_URL[url].append(q)

def show_table(rows,columns,limit=12):
    head="".join(f"<th>{html.escape(c)}</th>" for c in columns)
    body="".join("<tr>"+"".join(f"<td>{html.escape(str(r.get(c,'')))}</td>" for c in columns)+"</tr>" for r in list(rows)[:limit])
    display(HTML(f"<div style='overflow:auto;max-height:520px'><table style='font-size:13px'><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>"))

show_table([{"형식":k,"문서 수":v} for k,v in PROFILE["format"].most_common()], ["형식","문서 수"],20)
show_table([{"학위과정 초안":k,"문서 수":v} for k,v in PROFILE["degree"].most_common()], ["학위과정 초안","문서 수"])

문서 54,540건, 출처 URL 100,248개


형식,문서 수
.html,35538
.hwp,5918
.pdf,3757
.jpg,3755
.png,3347
.zip,678
.docx,552
.xlsx,447
.ole,296
.hwpx,172


학위과정 초안,문서 수
unknown,44417
graduate,9947
undergraduate,176


## 3. 본문·FAQ·표 분리
기존 FAQ 구조화 자료를 우선 사용합니다. FAQ는 질문과 답변을 함께 보존하고, 안내/규정은 제목·조항의 경계로 나눕니다. 게시판 목록은 상세 글과 구분합니다. 표는 단독으로 분류하지 않고 부모 문서의 대상과 주제를 붙입니다.

`REPARSE_SELECTED_HTML=True`이면 추출이 의심스러운 HTML만 저장된 원본에서 읽습니다. 새 수집이나 기존 수집 프로그램 실행은 하지 않습니다.

In [3]:
HEADING_RX = re.compile(r"^제\s*\d+조(?:의\d+)?(?:\s|\()|^부칙|^경과조치|^별표|^별지|^(정의|대상|신청시기|신청절차|신청방법|신청기간|유의사항|졸업요건|이수요건|등록안내|제출서류)$")

def faq_pairs(d):
    rows = list(FAQ_BY_DOC.get(d.get("doc_id"),[]))
    for u in source_urls(d): rows.extend(FAQ_BY_URL.get(u,[]))
    unique={}
    for q in rows:
        if q.get("question") and q.get("answer"):
            key=(q["question"].strip(),q["answer"].strip())
            unique[key] = {"unit_type":"faq","heading":key[0],"text":f"질문: {key[0]}\n답변: {key[1]}"}
    if unique: return list(unique.values())
    lines=d.get("text","").splitlines(); pairs=[]; question=[];answer=[];state=None
    for line in lines:
        label=line.strip()
        if label == "Q":
            if question and answer: pairs.append({"unit_type":"faq","heading":" ".join(question),"text":"질문: "+" ".join(question)+"\n답변: "+" ".join(answer)})
            question=[];answer=[];state="q"
        elif label == "A": state="a"
        elif state=="q": question.append(label)
        elif state=="a": answer.append(label)
    if question and answer: pairs.append({"unit_type":"faq","heading":" ".join(question),"text":"질문: "+" ".join(question)+"\n답변: "+" ".join(answer)})
    return pairs

def board_header_signature(values):
    """게시글 목록의 제목 + 번호/작성일/조회수 등 메타데이터 열을 확인."""
    labels=[re.sub(r"[\s.\-_:/]+","",str(x)).lower() for x in values]
    has_title=any(x in {"제목","글제목","게시글제목","title","subject"} for x in labels)
    groups=[{"번호","순번","no","number"}, {"작성일","등록일","게시일","date","작성날짜"},
            {"조회수","조회","views","hits"}, {"작성자","글쓴이","등록자","writer","author"},
            {"파일","첨부","첨부파일","attachment"}]
    return has_title and sum(any(x in group for x in labels) for group in groups)>=2

def is_board_table(table):
    # 날짜·학점·신청기간 표와 구분하기 위해 단순 숫자/연도 대신 게시글 헤더를 사용.
    return bool(table) and any(board_header_signature(row) for row in table[:4])

def is_detail_document(d):
    u=urlsplit(d.get("source_url", ""))
    return bool(re.search(r"(?:^|&)(?:mode=view|command=view2|mod=document)(?:&|$)",u.query,re.I)
                or re.search(r"artclView|articleView",u.path,re.I))

def is_listing(d):
    # FAQ 목록 페이지에 실제 답변이 있으면 질문·답변별로 보존한다.
    if d.get("format")!=".html" or faq_pairs(d) or is_detail_document(d): return False
    title=d.get("title", "")
    table_signature=any(is_board_table(t) for t in d.get("tables",[]))
    lines=[x.strip() for x in d.get("text", "").splitlines()[:180] if x.strip()]
    text_signature=any(board_header_signature(lines[i:i+7]) for i in range(len(lines)))
    listing_title=bool(re.search(r"게시판.{0,12}목록|게시글.{0,12}목록|board.{0,8}list",title,re.I))
    # '게시판 -- 목록', 일반 공지사항/뉴스센터도 실제 게시글 표 구조가 있으면 탐색 목록.
    return table_signature or text_signature or (listing_title and any(x in lines for x in ["조회수","작성일","등록일","No."]))


def cleaned_body(d):
    text=d.get("text","")
    lines=[line.strip() for line in text.splitlines() if line.strip()]
    lines=[line for line in lines if not re.fullmatch(r"조회수\s*\d*|이전글|다음글|목록|인쇄|공유|닫기|첨부파일",line)]
    return "\n".join(lines)

def reparse_body(d):
    body=cleaned_body(d)
    if not REPARSE_SELECTED_HTML or d.get("format")!=".html" or is_listing(d): return body
    suspicious = len(body)<100 or "whole_document" in str(d.get("selector", "")) or "/WEB-INF/" in body
    if not suspicious or not d.get("raw_path"): return body
    p=(PROJECT_ROOT/d["raw_path"]).resolve()
    if not p.is_relative_to(PROJECT_ROOT) or not p.exists(): return body
    try:
        from lxml import html as lh
    except ImportError as e:
        raise ImportError("원본 HTML 재추출에는 lxml이 필요합니다. 설치하거나 REPARSE_SELECTED_HTML=False로 두세요.") from e
    tree=lh.fromstring(p.read_bytes())
    nodes=tree.xpath('//*[contains(@class,"b-content-box")]|//*[@id="jwxe_main_content"]|//*[@id="contents"]|//main')
    if not nodes: return body
    node=nodes[0]
    for e in node.xpath('.//script|.//style|.//nav|.//header|.//footer'): e.drop_tree()
    for e in node.xpath('.//p|.//li|.//br|.//tr|.//div'): e.tail="\n"+(e.tail or "")
    return "\n".join(x.strip() for x in node.text_content().splitlines() if x.strip()) or body

def make_units(d):
    pairs=faq_pairs(d)
    if pairs: return pairs
    body=reparse_body(d)
    if is_listing(d): return [{"unit_type":"listing","heading":d.get("title",""),"text":body}]
    if (d.get("metadata") or {}).get("document_type") == "rule":
        # PDF에서는 조문이 붙어서 추출될 수 있다. 조문 번호+괄호 제목과 부칙을 절 경계로 보존한다.
        body = re.sub(r"(?<!\n)(?=제\s*\d+조(?:의\d+)?\s*\(|부칙\s*[<(])", "\n", body)
    headings=set(d.get("headings") or []);parts=[];current=[];heading=d.get("title","")
    for line in body.splitlines():
        if (line in headings or HEADING_RX.match(line)) and current:
            parts.append({"unit_type":"text","heading":heading,"text":"\n".join(current)})
            current=[];heading=line
        current.append(line)
    if current: parts.append({"unit_type":"text","heading":heading,"text":"\n".join(current)})
    for i,table in enumerate(d.get("tables") or []):
        if table:
            parts.append({"unit_type":"table","heading":d.get("title",""),"table_index":i,"rows":table,"text":"\n".join("\t".join(str(x) for x in row) for row in table)})
    return parts or [{"unit_type":"empty","heading":d.get("title",""),"text":""}]

## 4. 출처·학부 적용 대상·시점 정리
출처가 같은 대학에 속한다는 사실과 문서의 적용 대상은 구분합니다. 제목·본문 앞부분·URL 경로·공식 조직 목록을 함께 사용합니다. 대상이 충돌하면 검토로 남깁니다. 날짜 후보만 보고 시행일·신청 기간을 확정하지 않습니다.

In [4]:
def organizations(urls):
    names=[]
    for url in urls:
        u=urlsplit(url)
        for host,prefix,name in ORGANIZATIONS:
            if u.hostname and u.hostname.lower()==host and (prefix=="/" or u.path==prefix or u.path.startswith(prefix+"/")):
                names.append(name);break
    return list(dict.fromkeys(names))

def context(d):
    urls=source_urls(d)
    parent=INDEX.get(d.get("parent_url"))
    if parent: urls+=source_urls(parent)
    names=organizations(urls)
    own= d.get("title", "")+" "+d.get("text",d.get("text_hint", ""))[:1200]
    paths=" ".join(urlsplit(u).path for u in urls)
    ug=bool(re.search(r"\[학부|학부생|학부.?재학생|학부.?졸업|학부.?학생|undergraduate|/bachelor/|undergraduate",own+" "+paths,re.I))
    grad_title=bool(re.search(r"대학원|석.?박사",d.get("title", "")))
    grad=grad_title or bool(re.search(r"\[대학원|대학원생|석.?박사|일반대학원|/graduate|graduate-notice",own+" "+paths,re.I))
    grad_host=any((urlsplit(u).hostname or "").split(".")[0] in GRAD_HOSTS for u in urls)
    scope="mixed" if ug and grad else "undergraduate" if ug else "graduate" if grad or grad_host else "unknown"
    if scope=="unknown" and names and not any("대학원" in n for n in names):
        scope="undergraduate_context"  # 학과 출처의 간접 근거; 문서 대상 확정과 구분
    # 학교 학칙/시행세칙은 학부의 기반 규정이면서 대학원 언급도 포함한다.
    # 문서 선별 문맥만 보정하며 청크의 학과/학적 상태를 all로 확정하지 않는다.
    if re.fullmatch(r"(?:규칙\s*:\s*)?이화여자대학교\s*학칙(?:시행세칙)?(?:\.pdf)?", d.get("title", "").strip()) and any(
            (urlsplit(u).hostname or "").endswith("ewha.ac.kr") for u in urls):
        scope = "undergraduate_context"
    if d.get("doc_id") in CONFIRMED_SCOPE: scope=CONFIRMED_SCOPE[d["doc_id"]]
    return {"scope":scope,"organizations":names,"parent":parent,"parent_doc_id":parent.get("doc_id") if parent else None}

def applicability(d,text):
    m=d.get("metadata",{})
    published=m.get("published_at_raw")
    year_candidates=sorted(set(re.findall(r"(?<!\d)(?:19|20)\d{2}(?!\d)",d.get("title","")+" "+text)))
    admission=[];semesters=[];deadlines=[];effects=[]
    for line in (d.get("title","")+"\n"+text).splitlines():
        line=line.strip()
        if re.search(r"학번|입학년도|입학연도",line) and re.search(r"(?:19|20)\d{2}",line): admission.append(line[:500])
        if re.search(r"(?:19|20)\d{2}.?학년도.{0,12}[12].?학기|(?:19|20)\d{2}[-.]?[12]학기",line): semesters.append(line[:500])
        if re.search(r"신청.?기간|제출.?기한|신청.?기한|마감",line): deadlines.append(line[:500])
        if re.search(r"시행|개정|폐지",line): effects.append(line[:500])
    confirmed=d.get("doc_id") in CONFIRMED_CURRENT_IDS
    return {"as_of_date":AS_OF_DATE.isoformat(),"published_at_raw":published or "unknown", "mentioned_years":year_candidates,
            "applicable_admission_year":m.get("applicable_admission_year","unknown"),"applicable_semester":m.get("applicable_semester","unknown"),
            "admission_evidence":list(dict.fromkeys(admission)),"semester_evidence":list(dict.fromkeys(semesters)),
            "deadline_evidence":list(dict.fromkeys(deadlines)),"effective_evidence":list(dict.fromkeys(effects)),
            "effective_status":"human_confirmed_current" if confirmed else "unknown",
            "temporal_review_required":not confirmed,
            "note":"후보 연도/문구는 적용 기간 확정값이 아닙니다. 과거 기한 공지와 상시 규정의 구분에는 추가 확인이 필요합니다."}

## 5. 학사 선별과 판정 근거
`keep`: 학사 관련 근거가 있는 자료, `review`: 적용 대상·품질·주제가 모호한 자료, `exclude`: 명확한 범위 밖 자료.

표·조건·예외 절은 부모 학사 안내의 문맥을 이어받습니다. FAQ의 진로 질문처럼 독립된 항목은 각각 판단합니다. `keep`도 시행 상태 검토 전까지 검색 청크는 초안입니다.

In [5]:

def nonacademic_boundary(document, parent_title=""):
    """교육과정이라는 단어보다 문서의 실제 업무 목적을 우선한다."""
    title = document.get("title", "")
    text = document.get("text", "")
    purpose = title + "\n" + parent_title
    # 제목/부모 안내에 명시된 이수 관계만 예외로 인정한다.
    credit_link = bool(re.search(r"학점.{0,12}인정|졸업.{0,12}(?:요건|필수)|교과목.{0,12}(?:이수|대체)|교직.{0,12}이수", purpose))
    journal_title = bool(re.search(r"학술지|교육과정평가연구|논문.{0,10}(?:투고|간행)|투고.{0,10}(?:규정|지침)", title))
    publishing = bool(re.search(r"간행|투고|윤리|편집|게재|출판", title + "\n" + text))
    if journal_title and publishing and not credit_link:
        return "exclude", "journal_publication", "학술지 투고·간행·연구윤리 업무; 학부 학사 교육과정과 구분"
    school_ground = credit_link or bool(re.search(r"(?:본교|이화여자대학교|이화여대).{0,25}(?:학점.{0,8}인정|졸업.{0,8}요건|필수.{0,8}이수)", text))
    if re.search(r"한국파스퇴르연구소|초고속대용량스크리닝|HCS.{0,20}교육", purpose, re.I) and not school_ground:
        return "review", "external_training", "외부 기관 교육 과정; 본교 학점·졸업 요건 연결 근거 미확인"
    # 장학생 선발 공고의 자격증 평가표는 자격 취득 안내가 아니다.
    # 제목/부모 제목의 장학 목적과 본문의 선발·지원 근거를 함께 확인한다.
    scholarship_selection = (
        bool(re.search(r"장학(?:금|생)", purpose))
        and bool(re.search(r"선발|신청자격|지원대상|평가기준|평가 기준", text))
        and bool(re.search(r"장학금|장학생", text))
    )
    if re.search(r"등록민간자격|민간(?:등록)?자격|한국소비자업무협회|소비자업무전문가", purpose + "\n" + text) and not (school_ground or scholarship_selection):
        return "review", "external_certificate", "외부 민간 자격 취득 기준; 본교 학사 이수 관계 미확인"
    return None


def navigation_table(unit):
    rows = unit.get("rows", [])
    return (unit.get("unit_type") == "table" and bool(rows) and
            all(len(row) == 2 and re.fullmatch(r"조회하기|다운로드|바로가기", str(row[1]).strip()) for row in rows))

def housing_purpose(d,u,ctx):
    # FAQ는 질문 자체로, 이름이 모호한 첨부는 부모 공지의 제목으로 목적을 확인한다.
    title=u.get("heading", "") if u["unit_type"]=="faq" else d.get("title", "")
    parent_title=(ctx.get("parent") or {}).get("title", "")
    own_housing=bool(HOUSING_RX.search(title))
    inherited_housing=bool(d.get("format")!=".html" and parent_title and HOUSING_RX.search(parent_title))
    if not own_housing and not inherited_housing: return None
    purpose=title if own_housing else parent_title
    # [장학] 태그나 '장학재단' 기관 이름은 장학금 지급 업무의 근거로 삼지 않는다.
    if HOUSING_ACADEMIC_RX.search(purpose):
        return "review", "기숙사 관련 목적과 학사/장학금 요건 혼재; 직접 학사 적용 여부 검토"
    if HOUSING_ACTION_RX.search(purpose):
        return "exclude", "기숙사·생활관 입주/생활 업무; 장학재단 기관명과 실제 장학 업무를 구분"
    return "exclude", "기숙사·생활관 안내로 학사 업무 범위 밖"

def classify(d,u,ctx):
    title=u["heading"] if u["unit_type"]=="faq" else d.get("title", "")
    text=u.get("text","");full=title+" "+text
    topics=[k for k,rx in TOPICS.items() if rx.search(full)]
    title_topics=[k for k,rx in TOPICS.items() if rx.search(title)]
    academic_exception=bool(EXCEPTION_RX.search(full))
    reasons=[]
    if u["unit_type"]=="listing": return "exclude",topics,["탐색용 게시판 목록; FAQ 본문은 별도 처리"]
    if u["unit_type"]=="table" and is_board_table(u.get("rows",[])):
        return "exclude",topics,["번호·제목·작성일·조회수 등 게시글 목록 표"]
    boundary = nonacademic_boundary(d, (ctx.get("parent") or {}).get("title", ""))
    if boundary: return boundary[0],topics,[boundary[2]]
    if d.get("format") == ".html" and navigation_table(u):
        return "exclude",topics,["조회·다운로드 메뉴 표; 본문 데이터 표와 구분"]
    housing=housing_purpose(d,u,ctx)
    if housing: return housing[0],topics,[housing[1]]
    if u["unit_type"]=="empty" or not text.strip(): return "review",topics,["본문 추출 없음"]
    if HARD_RX.search(title):
        if academic_exception: return "review",topics,["범위 밖 제목이 학점 인정·졸업 요건과 함께 등장"]
        return "exclude",topics,["채용·시설 등 범위 밖 제목"]
    if re.search(r"졸업.?후.?진로|졸업 후.{0,12}(일|취업)|참고.?사이트",title) and not academic_exception:
        return "exclude",topics,["학사 요건/절차가 아닌 진로·링크 안내"]
    if re.search(r"명단|합격자|선발결과|선발 결과|선발자",title): return "review",topics,["명단·개인정보 검토 필요"]
    if ctx["scope"]=="graduate": return "exclude",topics,["대학원 전용 적용 대상의 근거"]
    parsed_binary = d.get("format") in {".pdf", ".hwp", ".hwpx", ".ole", ".doc", ".docx"} and str(d.get("parse_status", "")).startswith("draft_")
    source_rule = (d.get("metadata") or {}).get("document_type") == "rule"
    if source_rule and ctx["scope"] in {"undergraduate", "undergraduate_context"} and (parsed_binary or d.get("format") == ".html"):
        return "keep", topics, ["학부 기반 규정의 조문·예외·부칙 문맥 보존; 대상·시행일 확정은 별도"]
    if not topics: return "exclude",[],["설정된 학사 주제의 근거 없음"]
    if ctx["scope"] in ("unknown","mixed"): reasons.append("학부 적용 대상 미확정 또는 학부·대학원 혼재")
    layout_only_flags = {"pdf_tables_and_layout_unverified", "hwp_table_structure_unverified", "hwp_embedded_objects_not_extracted", "office_table_structure_unverified"}
    blocking_flags = set(d.get("quality_flags") or []) - (layout_only_flags if parsed_binary else set())
    if blocking_flags: reasons.append("추출 품질 플래그: "+", ".join(sorted(blocking_flags)))
    if SOFT_RX.search(title) and not academic_exception: reasons.append("행사·진로 등 학사 외 목적과 혼재")
    if not title_topics and u["unit_type"]!="table" and not academic_exception: reasons.append("본문 키워드만 존재; 학사 목적 확인 필요")
    if len(text.strip())<40: reasons.append("본문이 짧아 추가 확인 필요")
    if reasons: return "review",topics,reasons
    return "keep",topics,["학사 주제와 학부 대상/학과 문맥 확인"]

def classify_document(d):
    ctx=context(d)
    if ctx["scope"]=="unknown" and ctx["parent"]:
        pctx=context(ctx["parent"])
        if pctx["scope"] not in ("unknown","mixed"): ctx["scope"]=pctx["scope"]
    rows=[]
    for index,u in enumerate(make_units(d)):
        decision,topics,reasons=classify(d,u,ctx)
        signature=hashlib.sha256((u["unit_type"]+"\n"+re.sub(r"\s+"," ",u.get("text","")).strip()).encode()).hexdigest()
        unit_id=f"{d['doc_id']}_{index:04d}_{signature[:10]}"
        rows.append({"unit_id":unit_id,"doc_id":d["doc_id"],"title":d.get("title", ""),"source_url":d.get("source_url"),
            "source_variants":d.get("source_variants",[]),"raw_path":d.get("raw_path"),"format":d.get("format"),
            "parent_url":d.get("parent_url"),"parent_doc_id":ctx["parent_doc_id"],"organizations":ctx["organizations"],"degree_scope":ctx["scope"],
            "decision":decision,"topics":topics,"reasons":reasons,"quality_flags":d.get("quality_flags",[]),
            "applicability":applicability(d,u.get("text", "")),"content_hash":signature,**u})
    # 일반 안내의 제목 없는 신청절차/유의사항 절은 학사 문서 문맥을 보존한다.
    if any(r["decision"]=="keep" for r in rows) and not faq_pairs(d) and not is_listing(d):
        parent_topics=sorted({t for r in rows if r["decision"]=="keep" for t in r["topics"]})
        for r in rows:
            if r["decision"]=="exclude" and r["reasons"]==["설정된 학사 주제의 근거 없음"] and r["text"].strip() and not SOFT_RX.search(r["heading"]) and not HARD_RX.search(r["heading"]):
                r.update(decision="keep",topics=parent_topics,reasons=["같은 학사 안내의 조건·예외·표 문맥 보존"])
    # 같은 학사 문서의 짧은 단계/조건도 보존한다. 길이만으로 생긴 review를 되돌린다.
    # 메뉴·제목만 있는 단위는 다음 DOM 품질 검수에서 판정한다.
    if any(r["decision"] == "keep" for r in rows) and not faq_pairs(d) and not is_listing(d):
        parent_topics = sorted({t for r in rows if r["decision"] == "keep" for t in r["topics"]})
        for r in rows:
            if r["decision"] == "review" and r["reasons"] == ["본문이 짧아 추가 확인 필요"] and r["text"].strip():
                r.update(decision="keep", topics=parent_topics, reasons=["같은 학사 문서의 짧은 신청 단계·조건 보존"])
    return rows

OVERRIDES = {}  # 표본 CSV를 사람이 채운 후 아래 셀에서 불러오기

def apply_override(row):
    override=OVERRIDES.get(row["unit_id"])
    if override:
        row={**row,"automatic_decision":row["decision"],"decision":override["human_decision"],
             "human_review_note":override["human_note"],"reasons":row["reasons"]+["사람의 판정 반영"]}
    return row

## 6. 표본 200개 검토
전체 입력을 읽되, 판정·형식별로 작은 표본만 유지합니다. 유형별로 최소 사례를 우선 뽑고 나머지는 고정 난수로 채웁니다. 희소 유형을 포함하므로 전체 비율을 추정하는 무작위 표본은 아닙니다.

아래 표는 12개만 표시합니다. 전체 표본은 CSV와 JSONL로 저장합니다. `human_decision`을 `keep/review/exclude`로 채우고 판정 이유를 `human_note`에 적으세요.

In [6]:
rng=random.Random(SEED)
reservoirs=defaultdict(list);seen=Counter();preview_counts=Counter()
for d in read_jsonl(INPUT):
    for row in classify_document(d):
        preview_counts[row["decision"]]+=1
        key=(row["decision"],row["format"],row["unit_type"])
        seen[key]+=1
        bucket=reservoirs[key]
        if len(bucket)<8: bucket.append(row)
        else:
            pos=rng.randrange(seen[key])
            if pos<8: bucket[pos]=row
# 층별 1건을 우선 확보하고, 나머지를 고정 난수로 선택한다.
first=[v[0] for _,v in sorted(reservoirs.items(),key=lambda x:str(x[0])) if v]
rest=[r for v in reservoirs.values() for r in v[1:]]
rng.shuffle(rest)
SAMPLE=(first+rest)[:SAMPLE_SIZE]
print("초안 판정 집계(문서가 아닌 FAQ/절/표 단위):",dict(preview_counts))
print("검토 표본:",len(SAMPLE),"개")
show_table([{ "ID":r["unit_id"],"판정":r["decision"],"제목":r["title"][:80],"구분":r["unit_type"],"주제":", ".join(r["topics"]),"학부 대상":r["degree_scope"],"이유":" / ".join(r["reasons"]),"본문":r["text"][:220]} for r in SAMPLE], ["ID","판정","제목","구분","주제","학부 대상","이유","본문"])

REVIEW_DIR=OUTPUT_BASE/"표본검토_통합_v4"
REVIEW_DIR.mkdir(parents=True,exist_ok=True)
SAMPLE_JSON=REVIEW_DIR/"sample_units.jsonl"
REVIEW_CSV=REVIEW_DIR/"review_200.csv"
if not SAMPLE_JSON.exists():
    with SAMPLE_JSON.open("x",encoding="utf-8") as f:
        for r in SAMPLE: f.write(json.dumps(r,ensure_ascii=False)+"\n")
if not REVIEW_CSV.exists():
    with REVIEW_CSV.open("x",encoding="utf-8-sig",newline="") as f:
        fields=["unit_id","doc_id","automatic_decision","title","source_url","format","topics","reasons","text_preview","human_decision","human_note"]
        writer=csv.DictWriter(f,fieldnames=fields);writer.writeheader()
        for r in SAMPLE:
            writer.writerow({"unit_id":r["unit_id"],"doc_id":r["doc_id"],"automatic_decision":r["decision"],"title":r["title"],"source_url":r["source_url"],"format":r["format"],"topics":" | ".join(r["topics"]),"reasons":" | ".join(r["reasons"]),"text_preview":r["text"][:1500],"human_decision":"","human_note":""})
print("표본 저장:",REVIEW_DIR)
print("기존 CSV는 보존합니다. 규칙을 바꿔 표본을 새로 만들 때 REVIEW_DIR 이름도 바꾸세요.")

초안 판정 집계(문서가 아닌 FAQ/절/표 단위): {'keep': 18237, 'exclude': 64790, 'review': 19804}
검토 표본: 200 개


ID,판정,제목,구분,주제,학부 대상,이유,본문
d6a98bf11dccb64e110a609e_0000_32d2dfd712,exclude,채용공고문.bmp,text,"졸업·교육과정, 증명·신청",graduate,채용·시설 등 범위 밖 제목,"RISTI KOICA YP 채용공고 (주)RISTI 미래전략연구소는 글로벌 국제개발협력을 선도하는 전문기관으로 개발도상국가 ODA, 국제기구 및 다자개발은행 프로젝트사업, 양자/다자 국제협력사업, 과학기술혁신을 기반으로 글로벌 지속가능발전에 기여하고자 합니다. RISTI에서 열정을 가지고 꿈과 비전을 함께 실현해 나갈 역량 있는 인재를 모집합니다. 채용 기관 모집기관 근무지 주소 RISTI"
3be8138d53b452779d5ffc79_0000_6c1205c0e0,exclude,(서식) 제출서류목록(입시전형).doc,text,"수강·학점·성적, 졸업·교육과정, 증명·신청",graduate,대학원 전용 적용 대상의 근거,"이화여자대학교 디자인대학원 제 출 서 류 목 록 지원과정 석사학위과정 수험번호 지원전공 성 명 구 분 제출여부 (O, X) 비 고 제출서류 목록 1부 ㆍ본 대학원 소정양식(디자인대학원 홈페이지에서 다운로드) 입학원서 1부 ㆍ인터넷 원서접수 후 출력본 연구계획서 1부 ㆍ본 디자인대학원 소정양식(디자인대학원 홈페이지>게시판>서식모음>연구계획서(입학전형)을 다운로드) 성적증명"
0e645eee1cf6d6ca73864ba5_0000_c600bd6ba3,exclude,(붙임문서2) 2025 G20 Youth Summit 한국 대표단 선발 영문에세이(양식).docx,text,,undergraduate_context,설정된 학사 주제의 근거 없음,"주제 : 아래 주제 中 택 1① 아프리카의 지속가능한 발전을 증진하기 위한 글로벌 차원의 호혜적 협력 방안 및 한국의 역할, ② 식량(농업)·보건·교육·무역투자·인프라·거버넌스·평화안정 등 현재 아프리카 발전을 위해 가장 시급히 해결해야 하는 우선 순위 과제 및 해결 방안 (지원자가 1~3개의 우선 순위 과제를 확인하고 선정 이유 설명 필요) (주제 삭제 금지)주제 : 아래 주제 中 택 1①"
30cfade5128b06d453e3b904_0000_181523f20f,exclude,본문 이미지: 2015 인문과학대학 신입생 오리엔테이션 출발시간 및 장소 안내,text,,undergraduate_context,설정된 학사 주제의 근거 없음,금화터널 이화•금란 고등학교 후문 으로 오세요~* 이화여자대학교 하니솔빌딩 세브란스병원 이화•금란 이화여대 중학교 연세대학교 부속이화 초등학교 신촌역 (국철) # 럭키아파트 현대백화점 홍익문고 미라보호텔 을 2번출구 8번출구 동교동 신촌역(2호선) 이대역(2호선) 아현동 서강대후문 서강대교 서강대학교
3a90567db6f7c61c609cd7ff_0001_bc3fbd0a31,exclude,E-Badge > FAQ 게시판목록 | 이화여자대학교 미래혁신센터,faq,,unknown,설정된 학사 주제의 근거 없음,"질문: E-벗 내에서는 E-Badge를 어떻게 관리할 수 있나요? 답변: 배지 수령 전 ☞ 마이페이지 > E-Badge > E-Badge 취득 현황에서 발급 대기 중인 배지(흑백)를 확인 후 [배지수령하기] 버튼을 누르세요. 배지가 정상적으로 수령되면, 흑백의 배지가 컬러 배지로 변경되며, 마이페이지 화면에서도 수령한 E-Badge를 바로 확인할 수 있어요. 배지 수령 후 ☞ 수령한 배지들을"
6adbe2868d2102f0bc0dbb6b_0000_de4144760b,exclude,대학원 공지사항 게시판목록 | 이화여자대학교 국어국문학과,listing,"학적, 수강·학점·성적, 등록·장학",graduate,탐색용 게시판 목록; FAQ 본문은 별도 처리,공지사항 전체 학사 논문 장학 기타 전체 전체 학사 논문 장학 기타 대학원 공지사항 No. 제목 작성자 작성일 첨부 공지 [공지] [26-2] 공인어학능력시험 성적표 제출 안내 공지 국어국문학과 관리자 26.07.31 2295 국어국문학과 관리자 2026-07-31 공지 [공지] 일반대학원 학점교류 관련 공통 안내 공지 국어국문학과 관리자 26.07.31 2149 국어국문학과 관리자 2026
606eaa0be8513efa43292533_0001_e822d185bc,exclude,이화여자대학교-금융감독원 공동 주최 「2024 국제 컨퍼런스」 및 「청년 채용설명회」,table,,graduate,대학원 전용 적용 대상의 근거,"구 분 1부 국제 컨퍼런스 2부 ① A.I. 경진대회 2부 ② 청년 채용설명회 시간 09:00∼12:30 13:00~16:00 16:00~18:00 주관 금융감독원 KB국민은행 본교 인재개발원 내용 해외 감독당국, 글로벌 금융사, 연구기관 및 금융 감독원이 저탄소 전환금융 관련 현황과 정책, 전략 등에 대해 발표 제6회 Future Finance A.I. Challenge 결선 및 시상식 진"
2b60b6bb81c5394cb7d75847_0000_cd96fa2e13,exclude,Flexer Server,text,,undergraduate_context,설정된 학사 주제의 근거 없음,식품영양학과 동물실험실 외부인 사용신청서 지도교수 (연구책임자) 소 속 직 위 신 청 인 주 민 등 록 번호(앞자리) 소 속 직 위 E- mail 내 선 핸 드 폰 위의 외부 신청인에 대한 이화여자대학교 식품영양학과 동물실험실의 이용 및 출입을 허가합니다. 년 월 일 동물실험실 연구책임자 (서명) 글로벌식품영양연구소 소장
21af5048dc98e4979340a0ff_0011_971cee4367,exclude,(붙임6)_2015년_국가연구장학금(인문사회계)_FAQ(학생용).hwp,faq,등록·장학,graduate,대학원 전용 적용 대상의 근거,"질문: 석사과정 때 연구장학금(인문사회계) 수혜받았을 경우, 박사과정 때는 못받나요? 답변: 수혜가능 합니다. 석사과정과 박사과정을 동시 재학시에는 박사과정만 가능하지만, 석사과정 이후 박사과정이시라면 대학자체선발기준에 의해 선발된 장학생의 경우 수혜 가능합니다. 13"
ecbe0a40eb2498de012074b3_0000_c396044798,exclude,붙임2. 자기소개서 및 학업계획서.hwp,text,,undergraduate,설정된 학사 주제의 근거 없음,"[첨부2] 자기소개서 및 학업계획서 신청자 성명 (서명) 자기소개 (가정환경, 봉사활동 내용 등 포함) 전공에 대한 관심 및 학업 계획, 미래 계획 등"


표본 저장: /Users/gohanjun/Desktop/SNU/빅데이터핀테크응용사례특강/이화여대_학사행정_RAG/10_학사선별/표본검토_통합_v4
기존 CSV는 보존합니다. 규칙을 바꿔 표본을 새로 만들 때 REVIEW_DIR 이름도 바꾸세요.


### 사람이 입력한 판정 불러오기
검토를 하지 않았다면 자동 판정 그대로 유지됩니다. 아래 지표는 라벨이 있는 표본에서만 계산하며, 유형별 표본이므로 전체 데이터의 정확도 추정값으로 해석하지 않습니다.

In [7]:
OVERRIDES={};confusion=Counter()
with REVIEW_CSV.open(encoding="utf-8-sig",newline="") as f:
    for r in csv.DictReader(f):
        label=r.get("human_decision", "").strip()
        if not label: continue
        if label not in {"keep","review","exclude"}: raise ValueError(f"잘못된 판정: {label}")
        if not r.get("human_note", "").strip(): raise ValueError("사람 판정에는 human_note가 필요합니다.")
        OVERRIDES[r["unit_id"]]={"human_decision":label,"human_note":r["human_note"]}
        confusion[(r["automatic_decision"],label)]+=1
if not OVERRIDES: print("사람이 판정한 표본이 없습니다. 정확도는 계산하지 않습니다.")
else:
    tp=confusion[("keep","keep")]
    predicted_keep=sum(n for (a,h),n in confusion.items() if a=="keep")
    human_keep=sum(n for (a,h),n in confusion.items() if h=="keep")
    print("검토 완료:",len(OVERRIDES),"개")
    print("표본 keep 정밀도:",round(tp/predicted_keep,3) if predicted_keep else "계산 불가")
    print("표본 keep 재현율:",round(tp/human_keep,3) if human_keep else "계산 불가")
    show_table([{"자동 판정":a,"사람 판정":h,"단위 수":n} for (a,h),n in confusion.items()],["자동 판정","사람 판정","단위 수"])

사람이 판정한 표본이 없습니다. 정확도는 계산하지 않습니다.


## 7. 전체 선별·청크 저장
현재 설정은 `full`입니다. 표본만 검토하려면 `sample`로 바꾸세요. 새 규칙/새 표본 검토를 반영하려면 `RUN_NAME`도 바꿉니다. 같은 이름으로 재실행하면 입력·설정이 같을 때만 체크포인트에서 이어집니다.

최대 1,800자를 목표로 문장·행 경계에서 청크를 나눕니다. 긴 표는 헤더와 문서 제목을 매 조각에 붙이고 원래 표는 학사 문서 출력에 보존합니다. 구분이 어려운 표는 품질 검토 표시가 붙습니다. 동일 본문도 학과·대상·부모·시점이 다르면 별도로 남깁니다.

In [8]:
# RUN_MODE / RUN_NAME은 처음의 실행 설정 셀에서만 지정합니다.

OUTPUT_NAMES={"keep":"academic_documents.jsonl","review":"review.jsonl","exclude":"excluded.jsonl","chunks":"academic_chunks.jsonl","duplicates":"duplicate_units.jsonl"}

def write_json_atomic(path,data):
    temporary=Path(str(path)+".tmp")
    temporary.write_text(json.dumps(data,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    os.replace(temporary,path)

def dedup_key(row):
    context_key={"doc_id":row["doc_id"],"hash":row["content_hash"],"organizations":row["organizations"],"scope":row["degree_scope"],"applicability":row["applicability"],"parent_doc_id":row["parent_doc_id"],"decision":row["decision"]}
    return hashlib.sha256(json.dumps(context_key,ensure_ascii=False,sort_keys=True).encode()).hexdigest()

def text_pieces(text):
    lines=[]
    for line in text.splitlines():
        if len(line)>CHUNK_MAX_CHARS:
            lines.extend(re.findall(r".{1,%d}" % CHUNK_MAX_CHARS,line))
        else: lines.append(line)
    current=[];size=0
    for line in lines:
        if size+len(line)+1>CHUNK_MAX_CHARS and current:
            yield "\n".join(current);current=[];size=0
        current.append(line);size+=len(line)+1
    if current: yield "\n".join(current)

def chunks_for(row):
    prefix=row["title"]+"\n"+row["heading"]
    if row["unit_type"]=="table" and row.get("rows"):
        table=row["rows"];header="\t".join(str(x) for x in table[0]);pieces=[];batch=[];size=0
        for values in table[1:]:
            raw_line="\t".join(str(x) for x in values)
            for line in text_pieces(raw_line):
                if size+len(line)>CHUNK_MAX_CHARS and batch:
                    pieces.append(header+"\n"+"\n".join(batch));batch=[];size=0
                batch.append(line);size+=len(line)+1
        if batch: pieces.append(header+"\n"+"\n".join(batch))
        if not pieces: pieces=[header]
    else: pieces=list(text_pieces(row["text"]))
    for i,piece in enumerate(pieces):
        yield {"chunk_id":row["unit_id"]+f"_c{i:04d}","unit_id":row["unit_id"],"doc_id":row["doc_id"],"title":row["title"],
            "source_url":row["source_url"],"source_variants":row["source_variants"],"raw_path":row["raw_path"],"parent_doc_id":row["parent_doc_id"],
            "organizations":row["organizations"],"degree_scope":row["degree_scope"],"topics":row["topics"],"unit_type":row["unit_type"],
            "text":prefix+"\n"+piece,"applicability":row["applicability"],"quality_flags":row["quality_flags"]+(["table_header_candidate_requires_review"] if row["unit_type"]=="table" else [])+(["oversize_context_requires_review"] if len(prefix)+len(piece)>CHUNK_MAX_CHARS*2 else []),
            "chunk_status":"draft_requires_applicability_review" if row["applicability"]["temporal_review_required"] else "selected_human_current_confirmed"}

def run_full():
    output=OUTPUT_BASE/RUN_NAME;output.mkdir(parents=True,exist_ok=True)
    params={"input":str(INPUT),"size":INPUT.stat().st_size,"mtime_ns":INPUT.stat().st_mtime_ns,"as_of":AS_OF_DATE.isoformat(),
        "topics":TOPIC_PATTERNS,"hard_noise":HARD_NOISE,"soft_noise":SOFT_NOISE,"exceptions":ACADEMIC_EXCEPTION,
        "confirmed_scope":CONFIRMED_SCOPE,"confirmed_current":sorted(CONFIRMED_CURRENT_IDS),"overrides":OVERRIDES,
        "aux_inputs":{str(p.relative_to(PROJECT_ROOT)):{"size":p.stat().st_size,"mtime_ns":p.stat().st_mtime_ns} for p in [org_path,faq_path] if p.exists()},
        "chunk_max":CHUNK_MAX_CHARS,"reparse_html":REPARSE_SELECTED_HTML,"code_version":RULES_VERSION,
        "housing_pattern":HOUSING_PATTERN,"housing_actions":HOUSING_ACTION_PATTERN,"housing_academic":HOUSING_ACADEMIC_PATTERN}
    fingerprint=hashlib.sha256(json.dumps(params,ensure_ascii=False,sort_keys=True).encode()).hexdigest()
    checkpoint=output/"checkpoint.json"
    state={"fingerprint":fingerprint,"input_offset":0,"documents_processed":0,"counts":{},"output_offsets":{},"stage":"running"}
    seen_keys={}
    if checkpoint.exists():
        state=json.loads(checkpoint.read_text())
        if state["fingerprint"]!=fingerprint: raise ValueError("설정/입력이 바뀌었습니다. RUN_NAME을 새 이름으로 바꾸세요.")
        if state.get("stage")=="complete": return json.loads((output/"summary.json").read_text())
    elif any((output/name).exists() for name in OUTPUT_NAMES.values()):
        raise FileExistsError("체크포인트 없는 기존 결과가 있습니다. RUN_NAME을 바꾸세요.")
    streams={}
    try:
        for key,name in OUTPUT_NAMES.items():
            p=output/name;stream=p.open("r+b" if p.exists() else "w+b")
            # 마지막 확정 체크포인트 이후의 부분 쓰기만 정리하여 중복 없이 재개한다.
            stream.truncate(state["output_offsets"].get(key,0));stream.seek(0,2);streams[key]=stream
        for name in [OUTPUT_NAMES["keep"],OUTPUT_NAMES["review"],OUTPUT_NAMES["exclude"]]:
            for row in read_jsonl(output/name): seen_keys[dedup_key(row)]=row["unit_id"]
        counts=Counter(state["counts"])
        def emit(key,row): streams[key].write((json.dumps(row,ensure_ascii=False)+"\n").encode())
        def save():
            for stream in streams.values(): stream.flush();os.fsync(stream.fileno())
            state.update(counts=dict(counts),output_offsets={k:s.tell() for k,s in streams.items()})
            write_json_atomic(checkpoint,state)
        save()  # 첫 체크포인트 이전에 중단되어도 재개할 수 있게 먼저 기록
        with INPUT.open("rb") as source:
            source.seek(state["input_offset"])
            while True:
                line=source.readline()
                if not line: break
                if not line.strip(): state["input_offset"]=source.tell();continue
                d=json.loads(line)
                for row in classify_document(d):
                    row=apply_override(row);key=dedup_key(row)
                    if key in seen_keys:
                        emit("duplicates",{**row,"duplicate_of":seen_keys[key]});counts["duplicates"]+=1;continue
                    seen_keys[key]=row["unit_id"];emit(row["decision"],row);counts[row["decision"]]+=1
                    if row["decision"]=="keep":
                        for chunk in chunks_for(row): emit("chunks",chunk);counts["chunks"]+=1
                state["documents_processed"]+=1;state["input_offset"]=source.tell()
                if state["documents_processed"]%CHECKPOINT_EVERY==0:
                    save();print(f"{state['documents_processed']:,}/{N_DOCUMENTS:,} 문서 처리",end="\r")
        save()
        summary={"stage":"complete","scope":"이화여대 전체 학부","documents_processed":state["documents_processed"],"unit_counts":dict(counts),
                 "as_of_date":AS_OF_DATE.isoformat(),"output_dir":str(output),"input_file":str(INPUT),"fingerprint":fingerprint,
                 "completed_at":datetime.now().astimezone().isoformat(),"human_labeled_samples":len(OVERRIDES),
                 "note":"선별 단위는 문서·FAQ·절·표로 문서 수와 다릅니다. 청크는 적용 조건 검토가 필요한 초안입니다."}
        write_json_atomic(output/"summary.json",summary)
        state["stage"]="complete";save()
        print("\n전체 실행 완료:",output)
        return summary
    finally:
        for stream in streams.values(): stream.close()

if RUN_MODE=="full":
    RESULT=run_full()
else:
    print("표본 검토 모드입니다. 전체 결과 저장은 RUN_MODE='full'로 변경 후 이 셀을 실행하세요.")
    RESULT={"stage":"sample_only","unit_counts":dict(preview_counts),"note":"전체 입력의 자동 판정 집계이며 표본만 저장한 상태"}

## 8. 1차 선별 결과

아래는 품질 검수 전 자동 판정 단위 수입니다. 최종 사용 결과는 뒤의 품질 검수 단계에서 저장합니다. 표본 모드에서는 전체 결과와 품질 검수본을 생성하지 않습니다.


In [9]:
counts=RESULT.get("unit_counts",{})
labels=[("keep","포함","#16836b"),("review","검토","#b07816"),("exclude","제외","#697686")]
maximum=max([counts.get(k,0) for k,_,_ in labels]+[1]);svg=[]
for i,(key,label,color) in enumerate(labels):
    value=counts.get(key,0);y=35+i*45;width=360*value/maximum
    svg.append(f'<text x="10" y="{y+16}" font-size="14">{label}</text><rect x="80" y="{y}" width="{width}" height="23" fill="{color}"/><text x="{90+width}" y="{y+16}" font-size="14">{value:,}</text>')
display(HTML('<svg xmlns="http://www.w3.org/2000/svg" width="620" height="195" role="img" aria-label="자동 학사 선별 판정별 단위 수"><text x="10" y="20" font-size="15">자동 선별 초안: FAQ·절·표 단위 수</text>'+''.join(svg)+'<text x="10" y="185" font-size="12">표본 검토 및 시행 상태 확인 전 초안</text></svg>'))
print(json.dumps(RESULT,ensure_ascii=False,indent=2))

{
  "stage": "complete",
  "scope": "이화여대 전체 학부",
  "documents_processed": 54540,
  "unit_counts": {
    "keep": 17972,
    "chunks": 23373,
    "exclude": 63622,
    "review": 18711,
    "duplicates": 2526
  },
  "as_of_date": "2026-10-06",
  "output_dir": "/Users/gohanjun/Desktop/SNU/빅데이터핀테크응용사례특강/이화여대_학사행정_RAG/10_학사선별/학부_학사_통합_v4",
  "input_file": "/Users/gohanjun/Desktop/SNU/빅데이터핀테크응용사례특강/이화여대_학사행정_RAG/03_정제문서/documents_unique.jsonl",
  "fingerprint": "9e3e28d9133bd89982da02fd5acebcdecfef3bbee24a4cc1e62b81fc7e0d03a5",
  "completed_at": "2026-10-06T16:59:22.034347+09:00",
  "human_labeled_samples": 0,
  "note": "선별 단위는 문서·FAQ·절·표로 문서 수와 다릅니다. 청크는 적용 조건 검토가 필요한 초안입니다."
}


## 9. 오분류·UI 품질 검수

같은 실행의 포함 자료에 후속 검수 규칙을 적용합니다. HTML 메뉴·푸터는 로컬 DOM으로 대조합니다. PDF/HWP는 본문 추출 상태와 원본 존재를 확인하고 레이아웃 플래그를 보존합니다. 규정의 조문·부칙을 보호합니다.

중간 검수 파일은 `<RUN_NAME>_검수`에 남고, 현선님에게 전달할 문서·청크는 뒤의 인계 단계에서 다시 저장합니다.


In [10]:
QUALITY_ENABLED = RUN_QUALITY_AUDIT and RUN_MODE == "full"
QUALITY_STATE = {}
FINAL_RESULT = None
FINAL_OUTPUT = None

if QUALITY_ENABLED:
    if RESULT.get("stage") != "complete":
        raise RuntimeError("전체 학사 선별이 완료된 뒤 품질 검수를 실행하세요.")
    try:
        from lxml import html as quality_html_dependency
    except ModuleNotFoundError as error:
        raise ModuleNotFoundError("품질 검수에는 lxml이 필요합니다. 노트북 커널에서 %pip install lxml 실행 후 다시 실행하세요.") from error
    QUALITY_INPUT = Path(RESULT["output_dir"]) / "academic_documents.jsonl"
    QUALITY_OUTPUT = OUTPUT_BASE / f"{RUN_NAME}_검수"
    if not QUALITY_INPUT.is_file():
        raise FileNotFoundError(f"전체 선별 결과가 없습니다: {QUALITY_INPUT}")
    QUALITY_STATE = {
        "__name__": "ewha_academic_quality_audit",
        "QUALITY_PROJECT_ROOT": PROJECT_ROOT,
        "QUALITY_INPUT_PATH": QUALITY_INPUT,
        "QUALITY_OUTPUT_PATH": QUALITY_OUTPUT,
        "QUALITY_CHUNK_MAX_CHARS": CHUNK_MAX_CHARS,
        "QUALITY_SCOPE_GATE": nonacademic_boundary,
        "QUALITY_NAVIGATION_TABLE": navigation_table,
    }
    print("품질 검수 입력:", QUALITY_INPUT)
    print("최종 결과 폴더:", QUALITY_OUTPUT)
else:
    print("품질 검수 건너뜀: 전체 처리와 RUN_QUALITY_AUDIT=True 설정이 필요합니다.")


품질 검수 입력: /Users/gohanjun/Desktop/SNU/빅데이터핀테크응용사례특강/이화여대_학사행정_RAG/10_학사선별/학부_학사_통합_v4/academic_documents.jsonl
최종 결과 폴더: /Users/gohanjun/Desktop/SNU/빅데이터핀테크응용사례특강/이화여대_학사행정_RAG/10_학사선별/학부_학사_통합_v4_검수


### 9.1. 설정·공통 함수

In [11]:
# 검수 로직은 QUALITY_STATE에서 실행되어 선별 함수/변수를 보존합니다.
if QUALITY_ENABLED:
    exec(compile(r'''
import collections
import hashlib
import json
import re
from datetime import datetime
from pathlib import Path
from urllib.parse import parse_qs, unquote, urlsplit

from lxml import html as lxml_html

PROJECT_ROOT = QUALITY_PROJECT_ROOT
AUDIT_VERSION = "2.1"
CHUNK_MAX_CHARS = QUALITY_CHUNK_MAX_CHARS
MANUAL_DOC_DECISIONS = {
    "26183c26ec675563aeb65ece": {
        "decision": "exclude", "reason": "원문 FAQ 7번에 일반대학원/한국학대학원 전일제 미취업 대학원생 대상이라고 명시. '학부 때 전공'은 직전 학위 이력 질문임."},
    "56021b667bd8fbafa8ce7315": {
        "decision": "exclude", "reason": "원문 전체 확인: 제약바이오협회 외부 GMP 유료 교육 광고. 회원사 교육비/수강일정이며 이화 학점 인정·필수 이수 근거 없음."},
}  # 에이전트가 원문을 직접 확인한 판정. 사람 정답표/정확도로 취급하지 않는다.
MANUAL_UNIT_DECISIONS = {}


def locate_root():
    if PROJECT_ROOT is not None:
        return Path(PROJECT_ROOT).expanduser().resolve()
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in [base, base / "이화여대_학사행정_RAG",
                          base / "빅데이터핀테크응용사례특강/이화여대_학사행정_RAG"]:
            if (candidate / "10_학사선별/학부_학사_v2/academic_documents.jsonl").is_file():
                return candidate.resolve()
    raise FileNotFoundError("PROJECT_ROOT에 이화여대_학사행정_RAG 경로를 설정하세요.")


def read_jsonl(path):
    with path.open(encoding="utf-8") as stream:
        for number, line in enumerate(stream, 1):
            if line.strip():
                try:
                    yield json.loads(line)
                except json.JSONDecodeError as error:
                    raise ValueError(f"{path.name} {number}행 JSON 오류") from error


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def norm(text):
    return re.sub(r"\s+", "", str(text)).casefold()


def plain(text):
    return " ".join(str(text).split())


def tokens(node):
    return [plain(t) for t in node.xpath(".//text()[not(ancestor::script or ancestor::style)]")
            if plain(t)]


ROOT = locate_root()
INPUT = Path(QUALITY_INPUT_PATH)
ORIGINAL = ROOT / "03_정제문서/documents_unique.jsonl"
OUTPUT = Path(QUALITY_OUTPUT_PATH)
print("입력:", INPUT)
print("검수본:", OUTPUT)
''', "학사_품질검수_단계_1", "exec"), QUALITY_STATE)


입력: /Users/gohanjun/Desktop/SNU/빅데이터핀테크응용사례특강/이화여대_학사행정_RAG/10_학사선별/학부_학사_통합_v4/academic_documents.jsonl
검수본: /Users/gohanjun/Desktop/SNU/빅데이터핀테크응용사례특강/이화여대_학사행정_RAG/10_학사선별/학부_학사_통합_v4_검수


### 9.2. 입력·원본 제목·메뉴 위치 확인

In [12]:
# 검수 로직은 QUALITY_STATE에서 실행되어 선별 함수/변수를 보존합니다.
if QUALITY_ENABLED:
    exec(compile(r'''
UNITS = list(read_jsonl(INPUT))
BY_DOC = collections.defaultdict(list)
for unit in UNITS:
    BY_DOC[unit["doc_id"]].append(unit)

# 원문 본문은 메모리에 보관하지 않는다. 제목·출처·파싱 상태만 보관한다.
ORIGINAL_INFO = {}
NOTICE_INDEX = collections.defaultdict(dict)
for document in read_jsonl(ORIGINAL):
    urls = [document.get("source_url", ""), document.get("final_url", "")]
    urls += [variant.get("source_url", "") for variant in document.get("source_variants", []) if isinstance(variant, dict)]
    for url in urls:
        parsed = urlsplit(url); query = parse_qs(parsed.query)
        article = query.get("articleNo", [None])[0]
        if article and query.get("mode", [""])[0] == "view" and "/attach/" not in parsed.path:
            title = document.get("title", "")
            if len(title) > 240:
                title = next((line.strip() for line in document.get("text", "").splitlines() if line.strip()), title[:240])
            NOTICE_INDEX[(parsed.hostname, article)][document["doc_id"]] = {"doc_id": document["doc_id"], "title": title, "source_url": url}
    if document.get("doc_id") in BY_DOC:
        ORIGINAL_INFO[document["doc_id"]] = {
            key: document.get(key) for key in ["title", "source_url", "raw_path", "selector",
                                              "metadata", "parse_status", "collected_at", "parent_url", "source_variants"]
        }

UI_ATTRIBUTE = re.compile(
    r"footer|header|(?:^|[\s_-])(?:gnb|lnb|snb|navi|nav|paging|pagination|breadcrumb|skipnavi)(?:$|[\s_-])"
    r"|k2wiz_GNB|path-navi|viewTop|b-btn-wrap|b-tab-box|tab-menu|tab-list|mMenu|_location", re.I)
UI_EXACT = re.compile(
    r"^(?:HOME|TOP|LIST|VIEW MORE|MORE|PRINT|프린트|인쇄|목록|목록으로|이전|다음|닫기|더보기|검색|공유|"
    r"본문바로가기|본문 바로가기|주메뉴바로가기|주메뉴 바로가기|메뉴열기|메뉴닫기|"
    r"캘린더 보기|리스트 보기|상단 이동|이전달 이동|다음달 이동|>다음달 이동|"
    r"(?:처음|이전|다음|마지막) 페이지로 이동하기|SNS 공유|"
    r"개인정보처리방침|네티즌윤리규약|이메일주소무단수집거부)$", re.I)
UI_RUNTIME = re.compile(r"^/WEB-INF/|^oia_sub$|^//<!\[CDATA\[|^//\]\]>$|^var (?:resPath|boardTy|boardNo|siteId|clientId)\s*=")
STRONG_ACADEMIC = re.compile(
    r"학칙|학사.?규정|수강.?신청|수강.?철회|성적.?정정|성적.?평가|학점.?인정|학점.?교류|"
    r"(?:최소|필수|이수|전공|교양).{0,10}학점|\d+\s*학점|교과(?:목|과정)|교육과정|"
    r"졸업.{0,12}(?:요건|기준|필수|시험|논문|연구|설계)|(?:전공|교직).{0,10}이수|"
    r"휴학|복학|재입학|자퇴|제적|등록금|장학금|장학생|학자금|학사일정|재학증명|성적증명")
CURRICULUM = re.compile(r"교과(?:목|과정)|교육과정|수강|\d+\s*학점|(?:최소|필수|이수|전공).{0,8}학점|졸업.{0,10}(?:요건|필수|시험|논문)|교직.{0,10}이수")
DIRECT_ACADEMIC_PURPOSE = re.compile(
    r"수강|학점|성적|학칙|학사|교육과정|교과(?:목|과정)|졸업.{0,12}(?:요건|기준|시험|논문|설계|연구)|"
    r"등록금|장학금.{0,18}(?:신청|선발|지원|지급)|장학생.{0,12}(?:신청|선발)|"
    r"휴학|복학|재입학|도전학기|TELOS.{0,12}(?:트랙|신청)|교환학생")
EVENT_PURPOSE = re.compile(r"경진대회|공모전|세미나|워크숍|특강|박람회|FAIR|졸업.{0,8}전시|졸업식.?후.?행사|장학증서수여식", re.I)
INTRO_PURPOSE = re.compile(r"(?:학과|학부|전공)\s*소개|인사말|연혁|교육목표|졸업생\s*현황")


def dom_info(unit):
    """DOM에서 실제 공지 제목·UI 위치·이미지 여부를 확인. 본문을 통째로 교체하지 않는다."""
    info = {"status": "missing", "actual_title": None, "ui_blocks": [], "ui_only_lines": [],
            "is_board_list": False, "body_images": 0, "body_preview": "", "raw_sha256": None}
    raw_path = unit.get("raw_path")
    if not raw_path:
        return info
    path = (ROOT / raw_path).resolve()
    if not path.is_relative_to(ROOT) or not path.is_file():
        return info
    try:
        data = path.read_bytes()
        info["raw_sha256"] = hashlib.sha256(data).hexdigest()
        if unit.get("format") != ".html":
            original = ORIGINAL_INFO.get(unit["doc_id"], {})
            info["status"] = "parsed" if str(original.get("parse_status", "")).startswith("draft_") and unit.get("text", "").strip() else "binary_extraction_unverified"
            info["body_preview"] = " ".join(u.get("text", "") for u in BY_DOC[unit["doc_id"]])[:12000]
            info["body_heading"] = unit.get("heading", "")[:240]
            info["parent_notices"] = []
            info["content_format"] = unit.get("format")
            return info
        root = lxml_html.fromstring(data)
    except (OSError, ValueError, lxml_html.etree.ParserError) as error:
        info["status"] = "error: " + type(error).__name__
        return info
    info["status"] = "parsed"
    detail = is_detail_url(unit["source_url"])
    title_nodes = []
    if detail:
        title_nodes = root.xpath("//*[contains(concat(' ',normalize-space(@class),' '),' viewdata ')]/dt"
                                 "|//*[contains(concat(' ',normalize-space(@class),' '),' b-title-box ')]"
                                 "|//*[contains(concat(' ',normalize-space(@class),' '),' kboard-title ')]//h1"
                                 "|//*[contains(concat(' ',normalize-space(@class),' '),' kboard-title ')]//h2")
    for node in title_nodes:
        candidate = plain(" ".join(tokens(node)))
        if 4 <= len(candidate) <= 240 and candidate not in {"공지사항", "제목"}:
            info["actual_title"] = candidate
            break
    ui_nodes = []
    for element in root.iter():
        if not isinstance(element.tag, str):
            continue
        attrs = " ".join([element.get("id", ""), element.get("class", ""), element.get("role", "")])
        if element.tag in {"nav", "header", "footer"} or element.get("role") == "navigation" or UI_ATTRIBUTE.search(attrs):
            ui_nodes.append(element)
        class_tokens = set((element.get("class") or "").split())
        if {"board", "list"}.issubset(class_tokens) and not detail:
            info["is_board_list"] = True
    # 안쪽 UI를 중복 수집하지 않도록 최상위 UI 노드만 취한다.
    ui_set = set(ui_nodes)
    ui_nodes = [node for node in ui_nodes if not any(parent in ui_set for parent in node.iterancestors())]
    blocks = []; ui_lines = set()
    for node in ui_nodes:
        parts = tokens(node)
        if parts:
            blocks.append({"text": "\n".join(parts), "dom": node.tag + "#" + node.get("id", "") + "." + node.get("class", "")})
            ui_lines.update(norm(part) for part in parts)
    for node in ui_nodes:
        if node.getparent() is not None:
            node.drop_tree()
    content_lines = {norm(part) for part in tokens(root)}
    info["ui_blocks"] = blocks
    # 본문에 같은 문구가 있으면 삭제하지 않아 신청방법 같은 실제 항목 제목을 보호한다.
    info["ui_only_lines"] = sorted(ui_lines - content_lines)
    content_candidates = root.xpath("//*[@id='divView']|//*[contains(concat(' ',normalize-space(@class),' '),' b-content-box ')]"
                                    "|//*[contains(concat(' ',normalize-space(@class),' '),' kboard-content ')]"
                                    "|//*[@id='cms-content']|//main")
    content = content_candidates[0] if content_candidates else root
    info["body_images"] = len(content.xpath(".//img"))
    parts = tokens(content)
    info["body_preview"] = plain(" ".join(parts))[:12000]
    info["body_heading"] = next((part for part in parts if len(part) >= 4 and not re.fullmatch(r"<서식>|[\d\s\-]+", part)), "")[:240]
    original = ORIGINAL_INFO.get(unit["doc_id"], {})
    lineage_urls = [unit["source_url"], unit.get("parent_url", ""), original.get("parent_url", "")]
    for variant in original.get("source_variants") or []:
        if isinstance(variant, dict):
            lineage_urls.extend([variant.get("source_url", ""), variant.get("parent_url", "")])
    parents = {}
    for url in lineage_urls:
        if not url:
            continue
        parsed = urlsplit(url); article = parse_qs(parsed.query).get("articleNo", [None])[0]
        if article:
            for candidate in NOTICE_INDEX.get((parsed.hostname, article), {}).values():
                if candidate["doc_id"] != unit["doc_id"]:
                    parents[candidate["doc_id"]] = candidate
    info["parent_notices"] = list(parents.values())
    return info


def is_detail_url(url):
    parsed = urlsplit(url); query = parse_qs(parsed.query)
    return (query.get("mode", [""])[0].lower() == "view"
            or query.get("command", [""])[0].lower() in {"view", "view2"}
            or query.get("mod", [""])[0].lower() == "document"
            or bool(re.search(r"artclView|articleView", parsed.path, re.I)))


DOM = {}
for index, (doc_id, units) in enumerate(BY_DOC.items(), 1):
    DOM[doc_id] = dom_info(units[0])
print(f"포함 {len(UNITS):,}단위 / 원본 {len(BY_DOC):,}문서; HTML 확인 {sum(d['status']=='parsed' for d in DOM.values()):,}문서")
''', "학사_품질검수_단계_2", "exec"), QUALITY_STATE)


포함 17,972단위 / 원본 8,767문서; HTML 확인 8,767문서


### 9.3. 오분류·UI 정리 규칙

In [13]:
# 검수 로직은 QUALITY_STATE에서 실행되어 선별 함수/변수를 보존합니다.
if QUALITY_ENABLED:
    exec(compile(r'''
def clean_text(unit, info):
    if unit.get("format") != ".html":
        return unit["text"], []  # 바이너리 추출 본문은 HTML 메뉴 규칙으로 지우지 않는다.
    if unit["unit_type"] == "table":
        # 표는 셀/열/주석을 보존한다. UI 표인지 여부는 판정 함수에서 별도로 확인한다.
        return unit["text"], []
    ui_only = set(info["ui_only_lines"])
    kept, removed = [], []
    for line in unit["text"].splitlines():
        stripped = line.strip()
        reason = None
        if UI_EXACT.fullmatch(stripped):
            reason = "UI 조작 문구"
        elif UI_RUNTIME.search(stripped):
            reason = "템플릿/실행 코드 잔여물"
        elif stripped == "Incoming Exchange APPLY HERE" or re.fullmatch(r">\s*[^>]{1,160}", stripped):
            # 국제처의 breadcrumb 경로. 비교 연산/실제 수치 조건은 보호한다.
            if not re.search(r"\d|이상|이하|초과|미만", stripped):
                reason = "사이트 경로/상단 홍보 링크"
        elif re.fullmatch(r"\[이 게시물은 .+에 의해 .+에서 이동 됨\](?:\s*\[이 게시물은 .+\])*", stripped):
            reason = "게시물 이동 시스템 기록"
        elif norm(stripped) in ui_only:
            reason = "원본 DOM의 메뉴/푸터에만 존재하는 문구"
        if reason:
            removed.append({"text": line, "reason": reason})
        else:
            kept.append(line.rstrip())
    return "\n".join(kept).strip(), removed


def full_ui_block(text, info):
    normalized = norm(text)
    if not normalized:
        return None
    for block in info["ui_blocks"]:
        # 완전히 UI 내부에 있는 단위만 제거. 짧은 일반 단어는 제외한다.
        if len(normalized) >= 12 and normalized in norm(block["text"]):
            return block["dom"]
    return None


def review_decision(unit, cleaned, title, info, peers):
    key = unit["unit_id"]; doc_id = unit["doc_id"]
    if key in MANUAL_UNIT_DECISIONS or doc_id in MANUAL_DOC_DECISIONS:
        choice = MANUAL_UNIT_DECISIONS.get(key, MANUAL_DOC_DECISIONS.get(doc_id))
        return choice["decision"], "manual", choice["reason"]
    boundary = QUALITY_SCOPE_GATE({"title": title, "text": info.get("body_preview", "")},
                                  " ".join(p.get("title", "") for p in info.get("parent_notices", [])))
    if boundary: return boundary
    if unit.get("format") == ".html" and QUALITY_NAVIGATION_TABLE(unit):
        return "exclude", "navigation_table", "조회·다운로드 메뉴 표; 데이터 표와 구분"
    path = unquote(urlsplit(unit["source_url"]).path).lower()
    original_title = unit["title"]
    if unit["unit_type"] == "faq":
        question = norm(unit.get("heading", ""))
        if re.search(r"대학원|석사과정|박사과정", question) and not re.search(r"학부|학.?석사.?연계", question):
            return "exclude", "graduate_faq", "FAQ 질문의 직접 대상이 대학원/석사/박사 과정"
    viewer = bool(re.search(r"Flexer\s*Server", original_title, re.I))
    purpose_title = title
    if viewer:
        parent_titles = [item["title"] for item in info.get("parent_notices", [])]
        if len(set(parent_titles)) == 1:
            purpose_title = parent_titles[0]
        else:
            purpose_title = info.get("body_heading", title)
        compact_purpose = norm(purpose_title)
        first_context = norm(info["body_preview"][:300])
        ug_in_purpose = bool(re.search(r"학부|학.?석사.?연계|일반대학|사범대학", compact_purpose))
        grad_in_purpose = bool(re.search(r"대학원|석사|박사|석.박사", compact_purpose))
        if not parent_titles and re.search(r"(?:석사|박사).{0,15}(?:과정|학위)|석사/박사|석사‧박사", first_context[:100]):
            grad_in_purpose = True
        if re.search(r"대학원생을대상으로|대학원재학생을대상", norm(info["body_preview"])) and not re.search(
                r"학부.{0,12}(?:재학생|학생|학점등록생|정규등록생)", compact_purpose):
            grad_in_purpose = True
        if grad_in_purpose and not ug_in_purpose:
            return "exclude", "graduate_attachment", "첨부의 원문 제목/연결 공지에서 대학원 전용 문서 확인: " + purpose_title[:160]
        if re.search(r"채용|입사지원서|신입사원|이력서|취업", compact_purpose):
            if not re.search(r"학점|현장실습|교과|장학", compact_purpose):
                return "exclude", "career_attachment", "첨부의 목적이 채용/입사지원: " + purpose_title[:160]
        if re.search(r"기숙사|생활관|연합생활관", compact_purpose) and re.search(r"입주|입사|퇴사|모집|배정", compact_purpose):
            if not re.search(r"장학금|장학생|휴학|복학|등록금", compact_purpose):
                return "exclude", "housing_attachment", "첨부의 목적이 생활관 입주 업무: " + purpose_title[:160]
    if re.search(r"/(?:index|main)\.(?:do|html?)$", path):
        return "exclude", "homepage", "홈페이지 첫 화면의 공지·홍보 모음; 독립 학사 문서가 아님"
    if unit["unit_type"] != "faq" and (info["is_board_list"] or
            (re.search(r"게시판.{0,8}목록|게시글.{0,8}목록", original_title) and
             not is_detail_url(unit["source_url"]) and re.search(r"작성자|작성일|조회수|다음 페이지", unit["text"]))):
        return "exclude", "board_list", "게시글/졸업작품 목록과 페이지 이동 UI; 안내 본문이 아님"
    if unit["unit_type"] == "table":
        labels = [plain(row[0]) for row in unit.get("rows", []) if row]
        if labels and all(re.fullmatch(r"이전\s*글|다음\s*글", label) for label in labels):
            return "exclude", "adjacent_posts", "다른 게시물로 이동하는 이전글·다음글 표"
    if ui_location := full_ui_block(unit["text"], info):
        return "exclude", "dom_menu", "원본 DOM에서 메뉴/푸터 영역 확인: " + ui_location
    if not cleaned:
        return "exclude", "ui_only", "본문 전체가 UI/템플릿 문구"
    if (ORIGINAL_INFO.get(doc_id, {}).get("metadata") or {}).get("document_type") == "rule" and info["status"] == "parsed":
        if unit.get("format") != ".html" or re.match(r"^제\s*\d+조|^부칙|^경과조치|^별표|^별지", unit.get("heading", "")):
            return "keep", "regulation_context", "규정 조문·표·예외·부칙 원문 보존; 적용 조건은 인계 단계에서 근거만 구조화"
    if re.search(r"/(?:galleries|gallery|research-performance|story)(?:/|\.|$)", path):
        if DIRECT_ACADEMIC_PURPOSE.search(title[:200]) and STRONG_ACADEMIC.search(cleaned):
            return "review", "mixed_news", "연구·갤러리·수기 출처에 학사 요건 혼재; 직접 안내 여부 확인 필요"
        return "exclude", "research_gallery_story", "연구성과·갤러리·학생 수기; 학사 규칙/절차가 아님"
    if re.search(r"/news/(?:ewha-news|people|press)", path):
        return "review", "news_article", "학교 뉴스 기사; 직접 학사 안내인지 확인 필요"
    if re.search(r"/(?:information|notice-board)/(?:news|news-event)\.(?:do|html?)$", path):
        if DIRECT_ACADEMIC_PURPOSE.search(title[:200]):
            return "review", "mixed_department_news", "학과 소식 출처의 학사 정보; 직접 안내 여부 확인 필요"
        return "exclude", "department_news", "학과 뉴스·수상·동문 소식; 학사 기준/절차가 아님"
    if re.search(r"졸업생.{0,40}(?:현황|승진|임용|배출)|동문소식|장학기금.{0,15}전달|(?:연구팀|교수팀).{0,20}(?:발표|규명)", title[:240]):
        return "exclude", "news_profile", "졸업생/연구성과/기부 소식; 학사 규정·신청 안내가 아님"
    if re.search(r"\[수상\]|수상소식|(?:우수상|회장상|총장상).{0,10}수상", title[:240]) and not re.search(r"신청|제출|이수요건|교과목|학점인정|장학금.{0,12}(?:신청|선발)", title[:200]):
        return "exclude", "award_news", "수상·연구 성과 소식이며 학사 기준/신청 절차가 아님"
    if re.search(r"연구실.{0,30}(?:연구원|학부연구생).{0,12}모집|학부연구생.{0,25}모집|연구원.{0,10}모집", title[:240]):
        if re.search(r"교과목|학점|현장실습|장학금", title[:240]):
            return "review", "mixed_research_recruitment", "연구원 모집과 학점·실습·장학 조건 혼재"
        return "exclude", "research_recruitment", "연구실 연구인력 모집이며 학사 규칙/이수 절차가 아님"
    if re.search(r"연구실.{0,12}인증제", title[:240]):
        return "exclude", "lab_facility_certification", "시설 안전 인증제; 학생 학점/졸업 인증제와 구분"
    if re.search(r"연구실.?안전|연구실.?정기교육", title[:240]):
        if "연구실안전관리담당자" in norm(cleaned) and not re.search(r"학부|실험실습교과목", cleaned):
            return "exclude", "lab_staff_training", "안전관리 담당자 교육 일정; 학부 수강자 필수 교육과 구분"
        if re.search(r"대학원생|대학원싱|연구활동종사자", cleaned) and not re.search(r"학부|교과목", cleaned):
            return "review", "lab_training_scope", "연구활동종사자/대학원 안전교육; 학부 적용 근거 확인 필요"
    if re.search(r"채용|뉴스모니터링|체험형.{0,5}인턴|신입사원", title[:240]):
        if re.search(r"현장실습|학점.{0,8}인정|수강|교과목|장학생", title[:240]):
            return "review", "mixed_career", "채용/진로와 학점·실습·장학 요건 혼재"
        return "exclude", "career_advert", "채용·인턴·진로 홍보; 지원자의 졸업 요건을 학사 업무로 오인"
    if re.search(r"기업형\s*비교과\s*현장실습|연구형\s*비교과\s*현장실습", title[:240]):
        return "review", "noncredit_internship", "비교과 현장실습; 학점 인정/필수 이수와 연결되는지 확인 필요"
    graduate_title = bool(re.search(r"\[대학원|일반대학원|석.?박사|대학원.{0,8}(?:입학|장학|수강|졸업|영어|학적)", title[:200]))
    undergraduate_title = bool(re.search(r"\[학부|학부생|학부.{0,10}(?:신청|졸업|재학)|학.?석사.?연계", title[:200]))
    if graduate_title and not undergraduate_title:
        return "exclude", "graduate_only", "복원한 원문 제목에서 대학원 전용 업무 확인"
    if "입학안내" in title and re.search(r"모집과정.{0,30}석사|모집과정.{0,30}박사", info["body_preview"][:180]) and not re.search(r"학부|학.?석사.?연계", title):
        return "exclude", "graduate_admission", "대학원 석사·박사 입학안내에서 지원 자격의 학사학위를 학부 업무로 오인"
    if re.match(r"입학안내|수시모집요강|정시모집요강", title) and not re.search(r"재입학|편입생.*학점", title):
        return "review", "admissions_context", "입시/모집 자료; 재학생 학사 업무에 필요한 부분 확인"
    if re.search(r"시설안내|학생회", title[:100]) and not DIRECT_ACADEMIC_PURPOSE.search(title[:200]):
        return "review", "student_life_context", "학생회/시설 안내에 학사 키워드 혼재; 실제 학사 내용 재분리 필요"
    if re.search(r"한국소비자업무협회|민간(?:등록)?자격증|전국.?에너지.?공동학점", title[:240]) and not re.search(r"본교.{0,12}학점|학점.{0,8}인정|수강신청", cleaned):
        return "review", "external_training", "외부 교육·자격 과정; 본교 학점/필수 이수 연결 근거 확인 필요"
    if EVENT_PURPOSE.search(title[:240]):
        # 행사 이름이 들어가도 실제 장학금 신청·교과목 수강 안내는 보존한다.
        scholarship_process = bool(re.search(r"장학금.{0,24}(?:신청|선발|지원|지급)|장학생.{0,18}(?:신청|선발)", title[:240]))
        course_process = bool(re.search(r"교과목|수강신청|학점인정|졸업요건|교육실습|실무실습", title[:240]))
        if not scholarship_process and not course_process:
            if STRONG_ACADEMIC.search(cleaned):
                return "review", "mixed_event", "행사·대회·특강 본문에 학사/장학 조건 혼재; 자동 적재 보류"
            return "exclude", "event_advert", "전시 초대·행사 홍보; 학사 이수/신청 안내가 아님"
    if INTRO_PURPOSE.search(title[:160]):
        if CURRICULUM.search(cleaned):
            if len(cleaned) > 700 and re.search(r"양성|교육목표|교육목적|역사|비전", cleaned):
                return "review", "mixed_introduction", "학과 소개와 교육과정 설명이 섞임; 학사 부분 재분리 필요"
        elif re.search(r"지도교수|학과정원|연락처|전화번호|이메일", cleaned):
            return "review", "organization_context", "학과 소개의 인원/연락처 정보; 학사 문의에 필요한 범위 확인"
        else:
            return "exclude", "pure_introduction", "학과/전공 홍보·연혁·진로 소개; 학사 기준 없음"
    lines = [line.strip() for line in cleaned.splitlines() if line.strip()]
    heading = unit.get("heading", "")
    title_label = title.split("|")[0].strip()
    # 파서가 문장 전체를 heading으로 만들기도 한다. heading과 같다는 이유만으로
    # 조문/예외/신청 절차/대상 연도 문장을 제거하지 않는다.
    informational = bool(re.search(r"[:：]|제\s*\d+조|^\d+[.)]|※|(?:\d{2,4}).{0,8}(?:학번|입학생|학년도|학기)|"
                                     r"신청.{0,8}(?:화면|선택|가능)|발급.{0,10}(?:방법|가능|제한)|예외|요건", cleaned))
    heading_only = unit["unit_type"] == "text" and len(cleaned) <= 70 and not informational and len(lines) <= 2 and all(
        norm(line) in {norm(heading), norm(title_label), norm(title)} for line in lines)
    if heading_only:
        meaningful_peers = any(other["unit_id"] != key and (other["unit_type"] in {"table", "faq"}
                               or len(other["text"].strip()) >= 100) for other in peers)
        if meaningful_peers:
            return "exclude", "heading_only", "동일 문서에 본문/표가 별도로 있는 빈 제목 단위"
        return "review", "missing_body", "제목만 추출됨; 이미지·첨부·본문 추출 누락 확인 필요"
    if unit["unit_type"] != "table" and len(cleaned) < 180 and re.search(
        r"첨부.{0,15}참고|붙임.{0,15}참고|첨부파일|포스터|이전글과 동일|자세한.{0,15}(?:첨부|붙임|파일)", cleaned) and not re.search(
        r"\d{1,2}[./월]\d{1,2}|\d+\s*학점|(?:제출|신청|접수|정정).{0,15}(?:방법|기간|절차|방문|우편)|^방법|구비서류", cleaned):
        return "review", "attachment_stub", "내용 대신 첨부/이전글 참조만 있음; 본문 근거 부족"
    if info["status"] != "parsed":
        return "review", "raw_unavailable", "원본 HTML 확인 불가; 메뉴/범위 검수가 불완전"
    return "keep", "pass", "범위·UI 규칙 검수 통과; 의미 전체/적용 시점 확정은 별도"
''', "학사_품질검수_단계_3", "exec"), QUALITY_STATE)


### 9.4. 검수 판정 적용

In [14]:
# 검수 로직은 QUALITY_STATE에서 실행되어 선별 함수/변수를 보존합니다.
if QUALITY_ENABLED:
    exec(compile(r'''
AUDITED = []; LOGS = []; REMOVED_UI = []
for unit in UNITS:
    info = DOM[unit["doc_id"]]
    title = info["actual_title"] or unit["title"]
    if re.search(r"Flexer\s*Server", title, re.I):
        candidate = info.get("body_heading", "")
        if 4 <= len(candidate) <= 160 and re.search(
                r"신청서|추천서|확인서|서약서|계획서|보고서|규정|안내|교육|장학|기준|동의서|논문|교과", norm(candidate)):
            title = candidate
    cleaned, removed = clean_text(unit, info)
    decision, code, reason = review_decision(unit, cleaned, title, info, BY_DOC[unit["doc_id"]])
    if decision not in {"keep", "review", "exclude"}:
        raise ValueError("판정은 keep/review/exclude 중 하나여야 합니다.")
    revised = dict(unit)
    revised["source_unit_id"] = unit["unit_id"]
    revised["original_title"] = unit["title"]
    revised["title"] = title
    revised["heading"] = title if unit.get("heading") == unit["title"] else unit.get("heading", "")
    revised["text"] = cleaned
    revised["decision"] = decision
    revised["audit"] = {"version": AUDIT_VERSION, "rule": code, "reason": reason,
                         "html_status": info["status"], "title_restored": title != unit["title"],
                         "removed_ui_lines": len(removed), "reviewer": "rule_based_with_targeted_agent_review",
                         "parent_notices": info.get("parent_notices", [])}
    revised["reasons"] = list(unit.get("reasons", [])) + [reason]
    if title != unit["title"] or cleaned != unit["text"]:
        digest = hashlib.sha256(cleaned.encode()).hexdigest()
        revised["content_hash"] = digest
        revised["unit_id"] = unit["unit_id"] + "_a" + hashlib.sha256((title + "\n" + cleaned).encode()).hexdigest()[:8]
    AUDITED.append(revised)
    LOGS.append({"source_unit_id": unit["unit_id"], "unit_id": revised["unit_id"], "doc_id": unit["doc_id"],
                 "source_url": unit["source_url"], "original_title": unit["title"], "title": title,
                 "decision": decision, "rule": code, "reason": reason,
                 "text_changed": cleaned != unit["text"], "title_changed": title != unit["title"],
                 "removed_ui_lines": len(removed)})
    if removed:
        REMOVED_UI.append({"source_unit_id": unit["unit_id"], "doc_id": unit["doc_id"],
                           "source_url": unit["source_url"], "decision": decision,
                           "removed_lines": removed, "original_text": unit["text"], "cleaned_text": cleaned})
COUNTS = collections.Counter(unit["decision"] for unit in AUDITED)
RULE_COUNTS = collections.Counter((log["decision"], log["rule"]) for log in LOGS)
print("검수 판정:", dict(COUNTS))
print("UI 수정 단위:", len(REMOVED_UI), "/ 제목 복원:", sum(log["title_changed"] for log in LOGS))
''', "학사_품질검수_단계_4", "exec"), QUALITY_STATE)


검수 판정: {'keep': 17002, 'exclude': 806, 'review': 164}
UI 수정 단위: 208 / 제목 복원: 349


### 9.5. 최종 청크 재생성·저장

In [15]:
# 검수 로직은 QUALITY_STATE에서 실행되어 선별 함수/변수를 보존합니다.
if QUALITY_ENABLED:
    exec(compile(r'''
def text_pieces(text):
    lines = []
    for line in text.splitlines():
        lines.extend([line[i:i + CHUNK_MAX_CHARS] for i in range(0, len(line), CHUNK_MAX_CHARS)] or [""])
    current = []; size = 0
    for line in lines:
        if size + len(line) + 1 > CHUNK_MAX_CHARS and current:
            yield "\n".join(current); current = []; size = 0
        current.append(line); size += len(line) + 1
    if current:
        yield "\n".join(current)


def make_chunks(unit):
    prefix = "\n".join(dict.fromkeys([unit["title"], unit.get("heading", "")])).strip()
    pieces = []
    if unit["unit_type"] == "table" and unit.get("rows"):
        header = "\t".join(str(value) for value in unit["rows"][0])
        pieces = list(text_pieces("\n".join("\t".join(str(value) for value in row) for row in unit["rows"][1:]))) or [""]
        pieces = [header + ("\n" + piece if piece else "") for piece in pieces]
    else:
        pieces = list(text_pieces(unit["text"]))
    for index, piece in enumerate(pieces):
        chunk = {key: unit.get(key) for key in ["unit_id", "source_unit_id", "doc_id", "title", "original_title",
                    "source_url", "source_variants", "raw_path", "parent_doc_id", "organizations", "degree_scope",
                    "topics", "unit_type", "applicability", "quality_flags", "audit"]}
        chunk["chunk_id"] = unit["unit_id"] + f"_qa_c{index:04d}"
        chunk["section"] = unit.get("heading", "")
        chunk["text"] = prefix + "\n" + piece
        chunk["chunk_status"] = "draft_requires_applicability_review"
        if unit["unit_type"] == "table":
            chunk["quality_flags"] = list(chunk.get("quality_flags") or []) + ["table_header_candidate_requires_review"]
        yield chunk


KEPT = [unit for unit in AUDITED if unit["decision"] == "keep"]
CHUNKS = [chunk for unit in KEPT for chunk in make_chunks(unit)]
SUMMARY = {
    "stage": "complete", "audit_version": AUDIT_VERSION,
    "completed_at": datetime.now().astimezone().isoformat(timespec="seconds"),
    "scope": "선별 포함 자료의 오분류·메뉴 잔여물 검수",
    "input_units": len(UNITS), "input_documents": len(BY_DOC),
    "unit_counts": dict(COUNTS), "chunks": len(CHUNKS),
    "kept_documents": len({unit["doc_id"] for unit in KEPT}),
    "ui_changed_units": len(REMOVED_UI),
    "ui_changed_kept_units": sum(log["decision"] == "keep" and log["text_changed"] for log in LOGS),
    "removed_ui_lines": sum(log["removed_ui_lines"] for log in LOGS),
    "titles_restored_units": sum(log["title_changed"] for log in LOGS),
    "rules": [{"decision": decision, "rule": code, "units": count}
              for (decision, code), count in sorted(RULE_COUNTS.items())],
    "limitations": ["전수 규칙/DOM 검수와 의심 유형의 직접 검토 결과이며 모든 단위의 수동 판독은 아님.",
                    "1차 선별에서 이미 제외·검토된 자료의 누락률은 이번 검수 대상이 아님.",
                    "적용 연도·시행 여부·학과 범위와 표의 첫 행이 실제 헤더인지는 미확정.",
                    "표·서식의 짧은 내용은 길이만으로 제외하지 않음."],
}
MANIFEST = {
    "version": AUDIT_VERSION, "input_sha256": sha256_file(INPUT),
    "original_sha256": sha256_file(ORIGINAL),
    "raw_files": [{"doc_id": doc_id, "sha256": info["raw_sha256"], "status": info["status"]}
                  for doc_id, info in DOM.items()],
    "manual_doc_decisions": MANUAL_DOC_DECISIONS, "manual_unit_decisions": MANUAL_UNIT_DECISIONS,
}
OUTPUT.mkdir(parents=True, exist_ok=True)


def write_jsonl(name, records):
    path = OUTPUT / name
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as stream:
        for record in records:
            stream.write(json.dumps(record, ensure_ascii=False) + "\n")
    temporary.replace(path)


write_jsonl("academic_documents.jsonl", KEPT)
write_jsonl("academic_chunks.jsonl", CHUNKS)
write_jsonl("review_after_audit.jsonl", (unit for unit in AUDITED if unit["decision"] == "review"))
write_jsonl("excluded_after_audit.jsonl", (unit for unit in AUDITED if unit["decision"] == "exclude"))
write_jsonl("unit_audit.jsonl", LOGS)
write_jsonl("removed_ui.jsonl", REMOVED_UI)
(OUTPUT / "summary.json").write_text(json.dumps(SUMMARY, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
(OUTPUT / "input_manifest.json").write_text(json.dumps(MANIFEST, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
''', "학사_품질검수_단계_5", "exec"), QUALITY_STATE)


### 9.6. 저장 건수·청크 연결 확인

In [16]:
# 검수 로직은 QUALITY_STATE에서 실행되어 선별 함수/변수를 보존합니다.
if QUALITY_ENABLED:
    exec(compile(r'''
EXPECTED_COUNTS = {
    "academic_documents.jsonl": len(KEPT), "academic_chunks.jsonl": len(CHUNKS),
    "review_after_audit.jsonl": COUNTS["review"], "excluded_after_audit.jsonl": COUNTS["exclude"],
    "unit_audit.jsonl": len(UNITS), "removed_ui.jsonl": len(REMOVED_UI),
}
CHECKS = {}
saved_units = {unit["unit_id"]: unit for unit in read_jsonl(OUTPUT / "academic_documents.jsonl")}
chunk_ids = set()
for name, expected in EXPECTED_COUNTS.items():
    actual = 0
    for record in read_jsonl(OUTPUT / name):
        actual += 1
        if name == "academic_chunks.jsonl":
            if record["unit_id"] not in saved_units or record["chunk_id"] in chunk_ids or not record["text"].strip():
                raise ValueError("청크 연결/ID/빈 본문 오류")
            chunk_ids.add(record["chunk_id"])
    CHECKS[name] = {"expected": expected, "actual": actual, "ok": actual == expected}
if not all(check["ok"] for check in CHECKS.values()) or sum(COUNTS.values()) != len(UNITS):
    raise ValueError("입출력 건수 불일치")
CHECKS["reconciliation"] = {"ok": sum(COUNTS.values()) == len(UNITS)}
CHECKS["unit_ids"] = {"ok": len(saved_units) == len(KEPT)}
CHECKS["scope"] = {"ok": all(unit["decision"] == "keep" for unit in saved_units.values())}
(OUTPUT / "validation.json").write_text(json.dumps(CHECKS, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("저장 및 JSONL/청크 연결 확인 완료:", OUTPUT)
print(json.dumps({key: SUMMARY[key] for key in ["input_units", "unit_counts", "chunks", "ui_changed_kept_units"]}, ensure_ascii=False, indent=2))
''', "학사_품질검수_단계_6", "exec"), QUALITY_STATE)


저장 및 JSONL/청크 연결 확인 완료: /Users/gohanjun/Desktop/SNU/빅데이터핀테크응용사례특강/이화여대_학사행정_RAG/10_학사선별/학부_학사_통합_v4_검수
{
  "input_units": 17972,
  "unit_counts": {
    "keep": 17002,
    "exclude": 806,
    "review": 164
  },
  "chunks": 22339,
  "ui_changed_kept_units": 154
}


## 10. 인계 단계의 기준

- 문서별 한 행과 청크별 한 행을 구분합니다. 기존 선별 단위 파일을 문서 파일로 그대로 전달하지 않습니다.
- 학과 출처·키워드·언급 연도만으로 적용 조건을 확정하지 않습니다. 복수 조건이 혼재하면 unknown과 근거를 남깁니다.
- 게시일은 시행일로 쓰지 않으며 과거 자료를 날짜만으로 삭제하지 않습니다.
- FAQ는 질문+답변을 한 청크로 보존합니다. 긴 FAQ/문장은 본문을 자르지 않고 토큰 상한 초과 시 DB 적재를 보류합니다.
- 조문 인용/부칙/동일 절의 연속 청크는 연결합니다. 확정하지 못한 외부 규정 인용은 별도 파일에 남깁니다.


## 11. 인계용 문서 재조립

In [17]:
from copy import deepcopy
from urllib.parse import parse_qs
from datetime import timezone
from zoneinfo import ZoneInfo
import uuid

HANDOFF_VERSION = "1.2"
SOURCE_TYPES = {"규정", "안내", "FAQ", "공지", "교육과정"}
HANDOFF_READY = QUALITY_ENABLED and bool(QUALITY_STATE.get("CHECKS"))


def handoff_source_type(original, units):
    kind = (original.get("metadata") or {}).get("document_type", "")
    title = original.get("title", "")
    if kind == "rule" or re.search(r"학칙|시행세칙|(?:학사|학점인정|교직).{0,8}규정", title):
        return "규정"
    if kind == "faq" or (units and all(u.get("unit_type") == "faq" for u in units)):
        return "FAQ"
    if re.search(r"교육과정|교과과정|교과목.{0,8}(?:소개|해설)|전공.{0,8}이수요건", title):
        return "교육과정"
    return "공지" if kind == "notice" else "안내"


def handoff_is_attachment(original):
    urls = source_urls(original)
    return original.get("format") not in {None, ".html"} or any(
        re.search(r"/attach/|attachNo=|mode=download|/download|Flexer", u, re.I) for u in urls)


def handoff_parent(original):
    if not handoff_is_attachment(original):
        return None, "not_attachment"
    urls = [original.get("parent_url")]
    urls += [v.get("parent_url") for v in original.get("source_variants", []) if isinstance(v, dict)]
    found = {INDEX[u]["doc_id"] for u in urls if u in INDEX and INDEX[u]["doc_id"] != original["doc_id"]}
    primary = INDEX.get(original.get("parent_url"), {}).get("doc_id")
    if primary in found:
        return primary, "resolved_primary" if len(found) > 1 else "resolved"
    if len(found) == 1:
        return next(iter(found)), "resolved"
    return None, "ambiguous" if found else "unresolved"


def handoff_parent_sources(original):
    if not handoff_is_attachment(original):
        return [], []
    urls = sorted({u for u in [original.get("parent_url")] +
                   [v.get("parent_url") for v in original.get("source_variants", []) if isinstance(v, dict)] if u})
    ids = sorted({INDEX[u]["doc_id"] for u in urls if u in INDEX and INDEX[u]["doc_id"] != original["doc_id"]})
    return ids, urls


def handoff_unit_order(unit):
    try:
        return int((unit.get("source_unit_id") or unit["unit_id"]).split("_")[1])
    except (ValueError, IndexError):
        return 10**9


def handoff_assemble(original, units):
    parts = []; seen = set(); tables = []
    for unit in sorted(units, key=handoff_unit_order):
        text = unit.get("text", "").strip()
        if not text:
            continue
        signature = re.sub(r"\s+", " ", text)
        if signature not in seen:
            heading = (unit.get("heading") or "").strip()
            parts.append((heading + "\n" if heading and heading not in text else "") + text)
            seen.add(signature)
        if unit.get("unit_type") == "table":
            tables.append({"section": unit.get("heading", ""), "rows": deepcopy(unit.get("rows", [])),
                           "source_unit_id": unit.get("source_unit_id", unit["unit_id"]),
                           "header_status": "requires_review"})
    parent_id, parent_status = handoff_parent(original)
    parent_ids, parent_urls = handoff_parent_sources(original)
    judgments = HANDOFF_JUDGMENTS[original["doc_id"]]
    omitted = [u for u in judgments if u["decision"] != "keep" and u["audit"]["rule"] not in
               {"dom_menu", "ui_only", "heading_only", "adjacent_posts", "board_list", "homepage", "navigation_table"}]
    earlier_omitted = HANDOFF_PRESELECTION_OMISSIONS.get(original["doc_id"], [])
    return {
        "doc_id": original["doc_id"],
        "title": next((u["title"] for u in units if u.get("audit", {}).get("title_restored")), original.get("title", "")),
        "text": "\n\n".join(parts),
        "source_url": original.get("source_url", ""),
        "source_type": handoff_source_type(original, units),
        "collected_at": original.get("collected_at"),
        "raw_path": original.get("raw_path"),
        "parent_doc_id": parent_id,
        "parent_link_status": parent_status,
        "parent_doc_ids": parent_ids,
        "parent_source_urls": parent_urls,
        "format": original.get("format"),
        "extraction_status": "partial" if omitted or earlier_omitted else "extracted_draft",
        "source_parse_status": original.get("parse_status"),
        "quality_flags": sorted(set(original.get("quality_flags") or []) | ({"partial_academic_document"} if omitted or earlier_omitted else set())),
        "metadata_review_required": True,
        "tables": tables,
        "source_unit_ids": [u.get("source_unit_id", u["unit_id"]) for u in units],
        "omitted_units": earlier_omitted + [{"unit_id": u["unit_id"], "decision": u["decision"], "reason": u["audit"]["reason"], "stage": "quality"} for u in omitted],
    }


if HANDOFF_READY:
    HANDOFF_UNITS = defaultdict(list)
    HANDOFF_JUDGMENTS = defaultdict(list)
    for unit in QUALITY_STATE["AUDITED"]:
        HANDOFF_JUDGMENTS[unit["doc_id"]].append(unit)
    for unit in QUALITY_STATE["KEPT"]:
        HANDOFF_UNITS[unit["doc_id"]].append(unit)
    HANDOFF_ORIGINALS = {}
    for original in read_jsonl(INPUT):
        if original["doc_id"] in HANDOFF_UNITS:
            HANDOFF_ORIGINALS[original["doc_id"]] = original
    # 선별 단계에서 보류/제외한 본문도 기록한다. 검수에 들어온 keep 단위만 보고 완전 추출로 판정하지 않는다.
    HANDOFF_PRESELECTION_OMISSIONS = defaultdict(list)
    for filename in ["review.jsonl", "excluded.jsonl"]:
        for unit in read_jsonl(Path(RESULT["output_dir"]) / filename):
            benign_reasons = {"조회·다운로드 메뉴 표; 본문 데이터 표와 구분", "탐색용 게시판 목록; FAQ 본문은 별도 처리", "번호·제목·작성일·조회수 등 게시글 목록 표"}
            if unit["doc_id"] in HANDOFF_ORIGINALS and not (set(unit.get("reasons", [])) & benign_reasons):
                HANDOFF_PRESELECTION_OMISSIONS[unit["doc_id"]].append({
                    "unit_id": unit["unit_id"], "decision": unit["decision"],
                    "reason": "; ".join(unit.get("reasons", [])), "stage": "selection"})
    HANDOFF_DOCUMENTS = {}
    for doc_id, units in HANDOFF_UNITS.items():
        if doc_id not in HANDOFF_ORIGINALS:
            raise ValueError(f"원본 문서 연결 없음: {doc_id}")
        document = handoff_assemble(HANDOFF_ORIGINALS[doc_id], units)
        if document["text"].strip():
            HANDOFF_DOCUMENTS[doc_id] = document
    print("문서별 재조립:", len(HANDOFF_DOCUMENTS), "개 (학사 단위 수와 구분)")
else:
    print("전체 선별·품질 검수가 완료되어야 인계 파일을 만들 수 있습니다.")


문서별 재조립: 8450 개 (학사 단위 수와 구분)


## 12. 근거가 있는 적용 조건과 날짜만 구조화

In [18]:
HANDOFF_DATE_RX = re.compile(r"(?<!\d)(?P<y>(?:19|20)\d{2})\s*(?:년\s*|[.\-/]\s*)(?P<m>\d{1,2})\s*(?:월\s*|[.\-/]\s*)(?P<d>\d{1,2})(?:\s*일|\s*\.)?")
HANDOFF_TERM_RX = re.compile(r"(?<!\d)((?:19|20)\d{2})\s*(?:학년도|년도|년)?\s*[-.]?\s*([12])\s*학기")
HANDOFF_ARTICLE_RX = re.compile(r"제\s*(\d+)\s*조(?:\s*의\s*(\d+))?")


def handoff_lines(text):
    return list(dict.fromkeys(s.strip() for s in text.splitlines() if s.strip()))


def handoff_unknown_admission():
    return {"status": "unknown", "from": None, "to": None}


def handoff_applicability(section, text, document):
    # 학과 사이트에서 수집했다는 사실만으로 해당 학과 전용이라고 확정하지 않는다.
    lines = handoff_lines(section + "\n" + text)
    dept = {"status": "unknown", "values": []}
    years = handoff_unknown_admission()
    student = {"status": "unknown", "values": []}
    basis = []; ambiguous = []
    dept_all = [s for s in lines if re.search(r"모든\s*학과|전체\s*학과|학과.{0,5}(?:관계없이|무관)|전\s*학과.{0,12}(?:적용|대상)", s) and not re.search(r"다만|예외|제외|단,", s)]
    org_names = {name for _, _, name in ORGANIZATIONS if name and re.search(r"학과|학부|전공|대학$", name) and "대학원" not in name}
    direct = []
    for line in lines:
        names = [name for name in org_names if name in line]
        if not names:
            continue
        direct_scope = (line == section or line == document["title"] or bool(re.search(r"^(?:적용\s*)?대상\s*[:：]|^적용\s*(?:학과|전공)|(?:학생|전공자|소속자).{0,12}(?:대상|적용)", line)))
        if direct_scope and not re.search(r"다만|예외|제외|달리|반면|비교", line):
            direct.append((sorted(names), line))
    dept_sets = {tuple(names) for names, _ in direct}
    if dept_all and not direct:
        dept = {"status": "all", "values": []}; basis += dept_all
    elif len(dept_sets) == 1 and not dept_all:
        dept = {"status": "specified", "values": list(next(iter(dept_sets)))}; basis += [s for _, s in direct]
    elif direct or dept_all:
        ambiguous.append("학과 조건이 여러 범위로 혼재"); basis += [s for _, s in direct] + dept_all

    year_lines = [s for s in lines if re.search(r"학번|입학\s*연도|입학\s*년도|입학생|입학자", s)]
    year_rules = []
    for line in year_lines:
        if re.search(r"모든\s*(?:학번|입학\s*연도)|입학\s*연도.{0,8}(?:무관|관계없이)", line) and not re.search(r"다만|예외|제외|단,", line):
            year_rules.append(("all", None, None)); continue
        if re.search(r"제외|예외|다만|단,", line):
            continue
        range_match = re.search(r"((?:19|20)\d{2})\s*(?:학년도|년도|년|학번)?\s*[~∼～–—-]\s*((?:19|20)\d{2})\s*(?:학년도|년도|년)?\s*(?:입학생|입학자|학번)", line)
        if range_match:
            start, end = map(int, range_match.groups())
            if start <= end: year_rules.append(("specified", start, end))
            continue
        corroborated = list(re.finditer(r"(?<!\d)(\d{2})\s*학번\s*[(（]\s*((?:19|20)\d{2})\s*학년도\s*[)）]", line))
        if len(corroborated) == 1 and int(corroborated[0][1]) == int(corroborated[0][2]) % 100:
            year = int(corroborated[0][2]); year_rules.append(("specified", year, year)); continue
        # 조건과 연도가 바로 연결된 경우만 확정. 행사 연도를 입학 연도로 쓰지 않는다.
        found = list(re.finditer(r"((?:19|20)\d{2})\s*(?:학년도|년도|년)?\s*(?:(이전|이후|이상|이하)\s*)?(입학생|입학자|학번)\s*(부터|이후|이상|이전|이하|까지)?", line))
        if len(found) != 1:
            continue
        m = found[0]; year = int(m[1]); bound = m[2] or m[4]
        if bound in {"부터", "이후", "이상"}: year_rules.append(("specified", year, None))
        elif bound in {"이전", "이하", "까지"}: year_rules.append(("specified", None, year))
        else: year_rules.append(("specified", year, year))
    unique = set(year_rules)
    if len(unique) == 1:
        status, start, end = next(iter(unique)); years = {"status": status, "from": start, "to": end}; basis += year_lines
    elif year_lines:
        ambiguous.append("입학 연도 조건이 복수이거나 범위 미확정"); basis += year_lines

    status_lines = [s for s in lines if re.search(r"^(?:신청\s*|지원\s*|적용\s*)?대상\s*[:：]|^대상자\s*[:：]|^적용\s*대상|외국인특별전형\s*입학생", s)]
    status_rules = []
    for line in status_lines:
        if re.search(r"모든\s*학생|전체\s*학생|학적\s*상태.{0,8}(?:무관|관계없이)", line) and not re.search(r"제외|다만|단,", line):
            status_rules.append(("all", ())); continue
        values = []
        if re.search(r"외국인특별전형\s*입학생", line): values.append("외국인특별전형 입학생")
        for word in ["재학생", "휴학생", "복학생", "편입생", "졸업예정자", "신입생"]:
            if word in line: values.append(word)
        if values and not re.search(r"제외|불가|아닌|다만|단,", line): status_rules.append(("specified", tuple(sorted(values))))
    unique = set(status_rules)
    if len(unique) == 1:
        status, values = next(iter(unique)); student = {"status": status, "values": list(values)}; basis += status_lines
    elif status_lines:
        ambiguous.append("학적/학생 대상 조건이 복수이거나 예외 혼재"); basis += status_lines
    # 같은 절의 별도 줄에 있는 예외도 기본 적용 범위와 함께 해석한다.
    restriction_lines = [s for s in lines if re.search(r"다만|예외|제외|단\s*[,：:]|적용하지|적용되지", s)]
    dept_restrictions = [s for s in restriction_lines if re.search(r"학과|학부|전공|대학", s) or any(name in s for name in org_names)]
    year_restrictions = [s for s in restriction_lines if re.search(r"학번|입학|(?:19|20)\d{2}\s*학년도", s)]
    student_restrictions = [s for s in restriction_lines if re.search(r"재학생|휴학생|복학생|편입생|신입생|졸업예정|외국인", s)]
    if dept_restrictions:
        dept = {"status": "unknown", "values": []}; basis += dept_restrictions
        ambiguous.append("학과 예외/제외 조건 별도 줄 확인; 범위 수동 검토")
    if year_restrictions:
        years = handoff_unknown_admission(); basis += year_restrictions
        ambiguous.append("입학 연도 예외/제외 조건 확인; 단일 범위로 확정하지 않음")
    if student_restrictions:
        student = {"status": "unknown", "values": []}; basis += student_restrictions
        ambiguous.append("학생 상태 예외/제외 조건 확인; 범위 수동 검토")
    return {"department": dept, "admission_year": years, "student_status": student,
            "basis": "\n".join(dict.fromkeys(basis)), "review_notes": ambiguous}


def handoff_iso_date(match):
    try:
        return date(int(match["y"]), int(match["m"]), int(match["d"])).isoformat()
    except ValueError:
        return None


def handoff_dates(section, text, original):
    dates = {"posted_at": None, "amended_at": None, "effective_from": None, "effective_until": None, "target_term": None}
    evidence = {key: [] for key in dates}
    raw_posted = (original.get("metadata") or {}).get("published_at_raw")
    if isinstance(raw_posted, str):
        values = {handoff_iso_date(m) for m in HANDOFF_DATE_RX.finditer(raw_posted)} - {None}
        if len(values) == 1:
            dates["posted_at"] = next(iter(values)); evidence["posted_at"] = [raw_posted]
    lines = handoff_lines(section + "\n" + text)
    candidates = defaultdict(set)
    for line in lines:
        for m in HANDOFF_DATE_RX.finditer(line):
            value = handoff_iso_date(m)
            if not value: continue
            before, after = line[max(0, m.start()-24):m.start()], line[m.end():m.end()+30]
            key = None
            if re.match(r"\s*(?:부터|에|로부터)?\s*(?:이\s*(?:규정|학칙|세칙)[은는]?\s*)?시행", after) or re.search(r"시행\s*(?:일|일자|시기)?\s*[:：]\s*$", before):
                key = "effective_from"
            elif re.match(r"\s*(?:까지|에)?\s*폐지", after) or re.search(r"폐지\s*(?:일|일자)?\s*[:：]\s*$", before):
                key = "effective_until"
            elif re.match(r"\s*개정", after) or re.search(r"개정\s*(?:일|일자)?\s*[:：]\s*$", before):
                key = "amended_at"
            elif re.search(r"<\s*개정\s*$", before):
                key = "amended_at"
            if key:
                candidates[key].add(value); evidence[key].append(line)
        terms = HANDOFF_TERM_RX.findall(line)
        if terms and re.search(r"시행|적용\s*시기", line):
            evidence["effective_from"].append(line)
        if terms and re.search(r"대상\s*학기|해당\s*학기|신청|수강|모집|등록|개강|학기.{0,12}대상", line):
            for year, semester in terms:
                candidates["target_term"].add(f"{year}-{semester}")
            evidence["target_term"].append(line)
    for key in ["amended_at", "effective_from", "effective_until", "target_term"]:
        if len(candidates[key]) == 1:
            dates[key] = next(iter(candidates[key]))
    # 여러 부칙/개정 이력의 날짜 중 최댓값을 현재 시행일로 추측하지 않는다.
    return dates, {k: list(dict.fromkeys(v)) for k, v in evidence.items() if v}, {
        k: sorted(v) for k, v in candidates.items() if len(v) > 1}


## 13. FAQ·표·조문을 보존한 청크와 관계 연결

In [19]:
def handoff_text_pieces(text):
    # 질문+답변은 별도 처리하고, 일반 절은 단락/문장 경계를 우선한다.
    parts = []; batch = []; size = 0
    for paragraph in text.splitlines():
        if not paragraph.strip(): continue
        pieces = [paragraph]
        if len(paragraph) > CHUNK_MAX_CHARS:
            pieces = re.split(r"(?<=[.!?。])\s+", paragraph)
        for piece in pieces:
            if size + len(piece) + 1 > CHUNK_MAX_CHARS and batch:
                parts.append("\n".join(batch)); batch = []; size = 0
            # 긴 문장/예외는 잘라 버리지 않고 초과 길이 플래그로 남긴다.
            batch.append(piece); size += len(piece) + 1
    if batch: parts.append("\n".join(batch))
    return parts


def handoff_chunks(document, original, units):
    result = []
    if document["source_type"] == "FAQ" and not all(u.get("unit_type") == "faq" for u in units):
        # 단일 질문이 원문 제목에 명시된 FAQ만 본문 전체를 한 답변으로 묶는다.
        # FAQ 모음 파일은 질문을 추측하지 않고 적재 보류한다.
        if re.search(r"[?？]|(?:나요|까요|습니까)\s*$", document["title"]) and document["extraction_status"] != "partial":
            paired = deepcopy(sorted(units, key=handoff_unit_order)[0])
            paired.update(unit_type="faq", heading=document["title"],
                          text="질문: " + document["title"] + "\n답변: " + document["text"])
            units = [paired]
    for unit in sorted(units, key=handoff_unit_order):
        text = unit.get("text", "").strip()
        if not text: continue
        if unit.get("unit_type") == "faq" or (document["source_type"] == "FAQ" and unit.get("unit_type") != "table"):
            pieces = [text]  # FAQ 본문을 길이로 잘라 질문과 답변을 떨어뜨리지 않는다.
        elif unit.get("unit_type") == "table" and unit.get("rows"):
            rows = unit["rows"]; header = "\t".join(str(x) for x in rows[0]); pieces = []; batch = []; size = len(header)
            for row in rows[1:]:
                line = "\t".join(str(x) for x in row)
                if size + len(line) + 1 > CHUNK_MAX_CHARS and batch:
                    pieces.append(header + "\n" + "\n".join(batch)); batch = []; size = len(header)
                batch.append(line); size += len(line) + 1
            pieces.append(header + ("\n" + "\n".join(batch) if batch else ""))
        else:
            pieces = handoff_text_pieces(text)
        section = (unit.get("heading") or document["title"]).strip()
        if len(section) > 240:
            section = section[:240]
        for index, piece in enumerate(pieces):
            dates, date_evidence, date_ambiguities = handoff_dates(section, piece, original)
            flags = set(document["quality_flags"])
            if document["source_type"] == "FAQ" and unit.get("unit_type") != "faq":
                flags.add("faq_pairing_requires_review")
            if len(piece) > CHUNK_MAX_CHARS: flags.add("oversize_semantic_unit")
            if unit.get("unit_type") == "table": flags.add("table_header_requires_review")
            if date_ambiguities: flags.add("multiple_date_evidence")
            result.append({
                "chunk_id": f"{unit['unit_id']}_h{HANDOFF_VERSION.replace('.', '')}_{index:04d}",
                "doc_id": document["doc_id"], "title": document["title"], "source_type": document["source_type"],
                "source_url": document["source_url"], "section": section, "text": piece,
                "applicability": handoff_applicability(section, piece, document),
                "dates": dates, "related_chunk_ids": [], "related_chunk_links": [],
                "date_evidence": date_evidence, "date_ambiguities": date_ambiguities,
                "collected_at": document["collected_at"], "raw_path": document["raw_path"],
                "parent_doc_id": document["parent_doc_id"],
                "source_unit_id": unit.get("source_unit_id", unit["unit_id"]),
                "unit_id": unit["unit_id"], "unit_type": unit.get("unit_type"),
                "extraction_status": document["extraction_status"], "quality_flags": sorted(flags),
                "metadata_review_required": True,
            })
    return result


def handoff_article_key(text):
    m = HANDOFF_ARTICLE_RX.match(text.strip())
    return f"{int(m[1])}:{int(m[2]) if m[2] else 0}" if m else None


def handoff_connect(chunks):
    by_doc = defaultdict(list); by_unit = defaultdict(list); unresolved = []
    for chunk in chunks:
        by_doc[chunk["doc_id"]].append(chunk); by_unit[chunk["unit_id"]].append(chunk)
    def connect(left, right, kind, basis):
        if left["chunk_id"] == right["chunk_id"]: return
        for a, b in [(left, right), (right, left)]:
            if b["chunk_id"] not in a["related_chunk_ids"]:
                a["related_chunk_ids"].append(b["chunk_id"])
                a["related_chunk_links"].append({"chunk_id": b["chunk_id"], "kind": kind, "basis": basis})
    for parts in by_unit.values():
        for left, right in zip(parts, parts[1:]):
            connect(left, right, "same_section_continuation", left["section"])
    for doc_id, parts in by_doc.items():
        article_heads = {}
        for chunk in parts:
            key = handoff_article_key(chunk["section"])
            if key: article_heads.setdefault(key, chunk)
        for chunk in parts:
            for m in HANDOFF_ARTICLE_RX.finditer(chunk["text"]):
                key = f"{int(m[1])}:{int(m[2]) if m[2] else 0}"
                preceding = chunk["text"][max(0, m.start()-35):m.start()]
                # 다른 규정에 대한 인용을 같은 문서의 조문으로 잘못 연결하지 않는다.
                external = re.search(r"([가-힣·\s]{2,30}(?:학칙|세칙|규정))\s*$", preceding)
                if external and re.sub(r"\s+", "", external[1]) not in re.sub(r"\s+", "", chunk["title"]):
                    unresolved.append({"chunk_id": chunk["chunk_id"], "reference": external[1]+" "+m[0], "reason": "external_regulation_reference_requires_review"}); continue
                if key in article_heads:
                    connect(chunk, article_heads[key], "explicit_article_reference", m[0])
                else:
                    unresolved.append({"chunk_id": chunk["chunk_id"], "reference": m[0], "reason": "referenced_article_not_in_export"})
        # 규정의 부칙은 문서 전체에 붙는 근거. 적용 연도는 각 청크의 원문대로 유지한다.
        if parts and parts[0]["source_type"] == "규정":
            appendices = [c for c in parts if re.match(r"^(?:부칙|경과조치)", c["section"])]
            for appendix in appendices:
                explicit_refs = list(HANDOFF_ARTICLE_RX.finditer(appendix["text"]))
                if explicit_refs:
                    continue  # 명시 조문은 위 인용 연결로 처리. 다른 조문 전체에 붙이지 않는다.
                if re.search(r"학번|입학생|학과|전공|다만|특정", appendix["text"]):
                    unresolved.append({"chunk_id": appendix["chunk_id"], "reference": appendix["section"],
                                       "reason": "addendum_scope_requires_review"})
                    continue
                for article in article_heads.values():
                    connect(appendix, article, "document_addendum", appendix["section"])
        for table in [c for c in parts if c["unit_type"] == "table"]:
            neighbors = [c for c in parts if c["unit_type"] != "table" and c["section"] == table["section"]]
            if neighbors: connect(table, neighbors[0], "table_same_section", table["section"])
    return unresolved


## 14. 적재 가능 자료와 보류 자료 분리

`documents.jsonl`과 `chunks.jsonl`에는 아래 기준을 통과한 자료만 넣습니다. 일부 본문 누락, 손상된 문자, 추출 실패, 미확정 FAQ 연결은 보류합니다. **적용 조건 `unknown`, 날짜 `null`, 표 레이아웃의 검토 표시만으로는 제외하지 않습니다.** 보류 자료와 이유는 별도 JSONL에 남습니다.

기본 25개는 유형·형식 확인용 표본이며 전체 정확도 추정용 무작위 표본이 아닙니다.

In [20]:
# 일반 결측과 본문 결함을 구분하는 동일 기준을 내보내기/DB 적재에서 사용한다.
BLOCKING_QUALITY_FLAGS = {"partial_academic_document", "faq_pairing_requires_review", "encoding_damage",
                          "empty_text", "extraction_failed", "missing_body", "raw_unavailable"}
QUALITY_GATE_VERSION = "1.1"


def handoff_ingestion_reasons(record, document=None):
    reasons = []
    flags = set(record.get("quality_flags") or [])
    if document:
        flags.update(document.get("quality_flags") or [])
        if document.get("extraction_status") in {"partial", "failed", "unparsed"}:
            reasons.append("document_incomplete_or_failed")
        if document.get("ingestion_status") == "quarantined":
            reasons.append("parent_document_quarantined")
    if record.get("extraction_status") in {"partial", "failed", "unparsed"}:
        reasons.append("document_incomplete_or_failed")
    if not record.get("text", "").strip(): reasons.append("empty_text")
    if record.get("text", "").count("\ufffd") >= 2: reasons.append("encoding_damage")
    reasons += sorted(flags & BLOCKING_QUALITY_FLAGS)
    if re.search(r"error|failed|unsupported|needs_ocr|preview_only|unparsed", str(record.get("source_parse_status") or "")):
        reasons.append("source_extraction_failed")
    # 업무 목적은 원문 문서 전체로 판단한다. 청크에서는 본문 결함만 개별 검사한다.
    # 청크의 일부 단어만으로 문서의 학사 목적을 다시 판정하지 않는다.
    if record.get("chunk_id") and document is None:
        reasons.append("missing_document_context")
        boundary = None
    else:
        boundary = nonacademic_boundary(document if document is not None else record)
    if boundary: reasons.append(boundary[1])
    # metadata_review_required 및 unknown/null은 허용한다. 미확인 값을 all로 바꾸지 않는다.
    return sorted(set(reasons))


def handoff_choose_documents(documents):
    if HANDOFF_MODE == "full": return sorted(documents)
    if HANDOFF_MODE != "sample": raise ValueError("HANDOFF_MODE는 sample 또는 full이어야 합니다.")
    if not 20 <= HANDOFF_SAMPLE_DOCUMENTS <= 30: raise ValueError("표본은 문서 20~30개로 지정하세요.")
    selected = []
    def add(doc_id):
        if doc_id not in documents: return
        needed = []
        parent = documents[doc_id]["parent_doc_id"]
        if parent in documents and parent not in selected: needed.append(parent)
        if doc_id not in selected and doc_id not in needed: needed.append(doc_id)
        if len(selected) + len(needed) <= HANDOFF_SAMPLE_DOCUMENTS: selected.extend(needed)
    # 중앙 규정, 보완한 증명서 절차, 실제 FAQ 쌍을 표본에 우선 포함한다.
    for doc_id in sorted(documents):
        d = documents[doc_id]
        if d["source_type"] == "규정" and "학칙" in d["title"] and d.get("format") == ".html":
            add(doc_id); break
    add("0118832698479d858a5e356a")
    for source_type in ["규정", "FAQ", "교육과정", "안내", "공지"]:
        candidates = [i for i in sorted(documents) if documents[i]["source_type"] == source_type]
        if candidates: add(candidates[0])
    buckets = defaultdict(list)
    for doc_id, d in sorted(documents.items()): buckets[(d["source_type"], d.get("format"))].append(doc_id)
    for group in sorted(buckets, key=str): add(buckets[group][0])
    for doc_id in sorted(documents):
        add(doc_id)
        if len(selected) >= HANDOFF_SAMPLE_DOCUMENTS: break
    if len(selected) < 20: raise ValueError("검수 통과 문서가 20개 미만입니다. 보류 사유를 확인하세요.")
    return selected


if HANDOFF_READY:
    HANDOFF_INGESTION_READY = {}; QUARANTINED_DOCUMENTS = []; QUARANTINED_CHUNKS = []
    for doc_id, document in HANDOFF_DOCUMENTS.items():
        # 이 셀을 다시 실행해도 이전 실행의 보류 상태를 상속하지 않는다.
        document.pop("ingestion_status", None)
        document.pop("ingestion_reasons", None)
        reasons = handoff_ingestion_reasons(document)
        if document["source_type"] == "FAQ":
            # 구조화 FAQ 또는 원문 제목이 질문인 완전한 문서만 허용한다.
            has_pairs = all(u.get("unit_type") == "faq" for u in HANDOFF_UNITS[doc_id])
            if not has_pairs and not re.search(r"[?？]|(?:나요|까요|습니까)\s*$", document["title"]):
                reasons.append("faq_pairing_requires_review")
        # 표본 선택 전에 모든 문서의 청크 결함을 확인한다.
        # 결함 문서는 이유를 기록해 전체 보류하고, 다른 문서는 계속 처리한다.
        if not reasons:
            candidate_chunks = handoff_chunks(document, HANDOFF_ORIGINALS[doc_id], HANDOFF_UNITS[doc_id])
            if not candidate_chunks:
                reasons.append("no_chunks")
            reasons.extend(reason for chunk in candidate_chunks
                           for reason in handoff_ingestion_reasons(chunk, document))
        document["ingestion_status"] = "quarantined" if reasons else "ready"
        document["ingestion_reasons"] = sorted(set(reasons))
        if reasons:
            QUARANTINED_DOCUMENTS.append(document)
        else:
            HANDOFF_INGESTION_READY[doc_id] = document
    HANDOFF_SELECTED_IDS = handoff_choose_documents(HANDOFF_INGESTION_READY)
    EXPORT_DOCUMENTS = [HANDOFF_INGESTION_READY[doc_id] for doc_id in HANDOFF_SELECTED_IDS]
    EXPORT_CHUNKS = []
    for doc_id in HANDOFF_SELECTED_IDS:
        document = HANDOFF_INGESTION_READY[doc_id]
        chunks = handoff_chunks(document, HANDOFF_ORIGINALS[doc_id], HANDOFF_UNITS[doc_id])
        for chunk in chunks:
            chunk.update(ingestion_status="ready", ingestion_reasons=[])
        EXPORT_CHUNKS.extend(chunks)
    UNRESOLVED_CHUNK_REFERENCES = handoff_connect(EXPORT_CHUNKS)
    EXPORT_DIR = OUTPUT_BASE / f"{RUN_NAME}_전달_{HANDOFF_MODE}"
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    print("전체 재조립 문서:", len(HANDOFF_DOCUMENTS), "/ 적재 기준 통과:", len(HANDOFF_INGESTION_READY), "/ 보류:", len(QUARANTINED_DOCUMENTS))
    print("인계 모드:", HANDOFF_MODE, "/ 문서:", len(EXPORT_DOCUMENTS), "/ 청크:", len(EXPORT_CHUNKS))


전체 재조립 문서: 8450 / 적재 기준 통과: 7163 / 보류: 1287
인계 모드: full / 문서: 7163 / 청크: 16863


## 15. 추출 실패 기록·요구 스키마로 저장

In [21]:
DOCUMENT_SCHEMA = {
    "type": "object", "required": ["doc_id", "title", "text", "source_url", "source_type", "collected_at", "raw_path", "parent_doc_id"],
    "properties": {"doc_id": {"type": "string"}, "title": {"type": "string"}, "text": {"type": "string", "minLength": 1},
        "source_url": {"type": "string"}, "source_type": {"enum": sorted(SOURCE_TYPES)},
        "collected_at": {"type": ["string", "null"]}, "raw_path": {"type": ["string", "null"]},
        "parent_doc_id": {"type": ["string", "null"]}, "ingestion_status": {"enum": ["ready"]}}, "additionalProperties": True,
}
STATUS_VALUES = ["specified", "all", "unknown"]
CHUNK_SCHEMA = {
    "type": "object", "required": ["chunk_id", "doc_id", "text", "section", "source_url", "title", "source_type", "applicability", "dates", "related_chunk_ids"],
    "properties": {"chunk_id": {"type": "string"}, "doc_id": {"type": "string"}, "text": {"type": "string", "minLength": 1},
        "section": {"type": "string"}, "source_url": {"type": "string"}, "title": {"type": "string"},
        "source_type": {"enum": sorted(SOURCE_TYPES)},
        "applicability": {"type": "object", "required": ["department", "admission_year", "student_status", "basis"],
            "properties": {
                "department": {"type": "object", "required": ["status", "values"], "properties": {"status": {"enum": STATUS_VALUES}, "values": {"type": "array", "items": {"type": "string"}}}},
                "admission_year": {"type": "object", "required": ["status", "from", "to"], "properties": {"status": {"enum": STATUS_VALUES}, "from": {"type": ["integer", "null"]}, "to": {"type": ["integer", "null"]}}},
                "student_status": {"type": "object", "required": ["status", "values"], "properties": {"status": {"enum": STATUS_VALUES}, "values": {"type": "array", "items": {"type": "string"}}}},
                "basis": {"type": "string"}}},
        "dates": {"type": "object", "required": ["posted_at", "amended_at", "effective_from", "effective_until", "target_term"],
                  "properties": {key: {"type": ["string", "null"]} for key in ["posted_at", "amended_at", "effective_from", "effective_until", "target_term"]}},
        "related_chunk_ids": {"type": "array", "items": {"type": "string"}}, "ingestion_status": {"enum": ["ready"]}}, "additionalProperties": True,
}


def handoff_write_jsonl(path, records):
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as stream:
        for record in records:
            stream.write(json.dumps(record, ensure_ascii=False) + "\n")
    temporary.replace(path)


def handoff_failures():
    # 정제 문서 파일에 실패가 빠져 있을 수 있으므로 수집 대장도 확인한다.
    manifest = PROJECT_ROOT / "02_원본/crawl_manifest.jsonl"
    sources = [INPUT] + ([manifest] if manifest.is_file() else [])
    seen = set()
    for path in sources:
        for record in read_jsonl(path):
            if record.get("format") not in {".pdf", ".hwp", ".hwpx", ".ole"}: continue
            status = str(record.get("parse_status") or "unparsed")
            failed = not str(record.get("text") or "").strip() or bool(re.search(r"error|failed|unsupported|needs_ocr|preview_only|unparsed", status))
            if not failed: continue
            key = record.get("doc_id") or record.get("source_url")
            if key in seen: continue
            seen.add(key)
            parent, parent_status = handoff_parent(record)
            yield {"doc_id": record.get("doc_id"), "title": record.get("title", ""), "source_url": record.get("source_url"),
                "format": record.get("format"), "collected_at": record.get("collected_at"), "raw_path": record.get("raw_path"),
                "parent_doc_id": parent, "parent_link_status": parent_status,
                "extraction_status": "failed", "source_parse_status": status,
                "failure_reason": record.get("parse_error") or record.get("error") or "empty_or_unsupported_extraction",
                "collection_status": record.get("collection_status"), "scope": "collection_wide_binary_extraction_log"}


def handoff_check_records(documents, chunks):
    documents_by_id = {d["doc_id"]: d for d in documents}
    doc_ids = {d["doc_id"] for d in documents}; chunk_ids = {c["chunk_id"] for c in chunks}
    if len(doc_ids) != len(documents) or len(chunk_ids) != len(chunks): raise ValueError("문서/청크 ID 중복")
    for d in documents:
        if d.get("ingestion_status") != "ready" or handoff_ingestion_reasons(d): raise ValueError("보류 문서가 인계 목록에 들어갔습니다.")
        if set(DOCUMENT_SCHEMA["required"]) - d.keys() or not d["text"].strip() or d["source_type"] not in SOURCE_TYPES:
            raise ValueError("문서 스키마/빈 본문 오류")
    for c in chunks:
        if c.get("ingestion_status") != "ready" or handoff_ingestion_reasons(c, documents_by_id.get(c["doc_id"])): raise ValueError("보류 청크가 인계 목록에 들어갔습니다.")
        if set(CHUNK_SCHEMA["required"]) - c.keys() or not c["text"].strip() or c["doc_id"] not in doc_ids:
            raise ValueError("청크 스키마/문서 연결 오류")
        if set(c["related_chunk_ids"]) - chunk_ids: raise ValueError("내보내지 않은 관련 청크 ID")
        for field in ["department", "admission_year", "student_status"]:
            condition = c["applicability"][field]
            if condition["status"] not in STATUS_VALUES: raise ValueError("적용 조건 상태 오류")
            if condition["status"] != "unknown" and not c["applicability"]["basis"]: raise ValueError("적용 조건의 원문 근거 없음")
        if c["unit_type"] == "faq" and not ("질문:" in c["text"] and "답변:" in c["text"]):
            raise ValueError("FAQ 질문·답변 연결 오류")
    try:
        from jsonschema import Draft202012Validator
    except ImportError:
        schema_validation = "required_fields_and_links_only; optional jsonschema not installed"
    else:
        for d in documents: Draft202012Validator(DOCUMENT_SCHEMA).validate(d)
        for c in chunks: Draft202012Validator(CHUNK_SCHEMA).validate(c)
        schema_validation = "passed"
    return {"json_schema": schema_validation, "documents": len(documents), "chunks": len(chunks), "ids_and_required_fields": "passed"}


if HANDOFF_READY:
    HANDOFF_CHECK = handoff_check_records(EXPORT_DOCUMENTS, EXPORT_CHUNKS)
    handoff_write_jsonl(EXPORT_DIR / "documents.jsonl", EXPORT_DOCUMENTS)
    handoff_write_jsonl(EXPORT_DIR / "quarantined_documents.jsonl", QUARANTINED_DOCUMENTS)
    # 청크 보류 파일은 생성기를 사용해 전체 보류 본문을 메모리에 중복 적재하지 않는다.
    def quarantine_chunks():
        for document in QUARANTINED_DOCUMENTS:
            doc_id = document["doc_id"]
            for chunk in handoff_chunks(document, HANDOFF_ORIGINALS[doc_id], HANDOFF_UNITS[doc_id]):
                chunk.update(ingestion_status="quarantined", ingestion_reasons=document["ingestion_reasons"])
                yield chunk
    handoff_write_jsonl(EXPORT_DIR / "quarantined_chunks.jsonl", quarantine_chunks())
    QUALITY_GATE_REPORT = {
        "version": QUALITY_GATE_VERSION, "scope": "all_reassembled_documents_before_handoff_sampling",
        "assembled_document_count": len(HANDOFF_DOCUMENTS), "ready_document_count": len(HANDOFF_INGESTION_READY),
        "quarantined_document_count": len(QUARANTINED_DOCUMENTS),
        "reasons": dict(Counter(r for d in QUARANTINED_DOCUMENTS for r in d["ingestion_reasons"])),
        "blocking_flags": sorted(BLOCKING_QUALITY_FLAGS),
        "allowed": ["unknown applicability", "null dates", "metadata_review_required", "table/layout review warnings"],
        "scope_exclusions": {"selection_or_quality": "review.jsonl, excluded.jsonl, audit logs"},
        "limitations": ["규칙 기반 통과는 의미 정확성의 전체 보장이 아님", "임베딩 토큰 수는 실제 모델 적재 단계에서 추가 확인"]}
    write_json_atomic(EXPORT_DIR / "quality_gate.json", QUALITY_GATE_REPORT)
    handoff_write_jsonl(EXPORT_DIR / "chunks.jsonl", EXPORT_CHUNKS)
    handoff_write_jsonl(EXPORT_DIR / "unresolved_references.jsonl", UNRESOLVED_CHUNK_REFERENCES)
    handoff_write_jsonl(EXPORT_DIR / "extraction_failures.jsonl", handoff_failures())
    external_parents = {parent for d in EXPORT_DOCUMENTS for parent in d.get("parent_doc_ids", [])
                        if parent not in HANDOFF_SELECTED_IDS}
    referenced = [{"doc_id": item["doc_id"], "title": item.get("title"), "source_url": item.get("source_url"), "status": "metadata_only_not_in_export"}
                  for item in {v["doc_id"]: v for v in INDEX.values() if v["doc_id"] in external_parents}.values()]
    handoff_write_jsonl(EXPORT_DIR / "referenced_documents.jsonl", referenced)
    write_json_atomic(EXPORT_DIR / "schema.json", {"version": HANDOFF_VERSION, "documents": DOCUMENT_SCHEMA, "chunks": CHUNK_SCHEMA,
        "semantics": {"specified": "원문에 조건 명시", "all": "해당 조건에 제한이 없음이 원문에 명시", "unknown": "미확인 또는 조건 혼재; 공통 규정으로 취급하지 않음",
            "dates": "게시일·개정일·시행일을 구분. 복수 날짜는 추측하지 않고 null 및 근거 후보 보존", "tables": "원래 셀/열/주석 보존; 첫 행 헤더 의미는 미확정"}})
    HANDOFF_SUMMARY = {
        "stage": "jsonl_complete", "version": HANDOFF_VERSION,
        "completed_at": datetime.now(ZoneInfo("Asia/Seoul")).isoformat(timespec="seconds"),
        "quality_gate": QUALITY_GATE_REPORT,
        "mode": HANDOFF_MODE, "document_count": len(EXPORT_DOCUMENTS), "chunk_count": len(EXPORT_CHUNKS),
        "source_types": dict(Counter(d["source_type"] for d in EXPORT_DOCUMENTS)),
        "applicability_statuses": {field: dict(Counter(c["applicability"][field]["status"] for c in EXPORT_CHUNKS)) for field in ["department", "admission_year", "student_status"]},
        "input_file": str(INPUT), "selection_fingerprint": RESULT["fingerprint"],
        "quality_input_sha256": QUALITY_STATE["MANIFEST"]["input_sha256"],
        "files": {name: {"sha256": QUALITY_STATE["sha256_file"](EXPORT_DIR / name), "bytes": (EXPORT_DIR / name).stat().st_size}
                  for name in ["documents.jsonl", "chunks.jsonl", "schema.json", "extraction_failures.jsonl", "quarantined_documents.jsonl", "quarantined_chunks.jsonl", "quality_gate.json"]},
        "record_checks": HANDOFF_CHECK, "embedding_and_db_status": "not_loaded",
        "limitations": ["규칙 기반 메타데이터 초안. 미확인 값은 unknown/null이며 수동 확정 전 적용 범위 보장 없음.",
                        "일부 본문 누락/추출 실패는 보류 파일에만 저장하고 정상 인계·적재 대상에서 제외.",
                        "PDF/HWP 표·레이아웃 검증 플래그 보존. 새 OCR/원본 재수집은 수행하지 않음.",
                        "FAQ는 구조화 쌍 또는 원문 제목에 명시된 단일 질문+전체 답변만 허용. 그 밖에는 보류.",
                        "여러 공지에서 재사용한 첨부는 원본 parent_url을 대표 연결로 사용하고 parent_doc_ids/parent_source_urls에 나머지를 보존.",
                        "수집 전체의 바이너리 추출 실패는 별도 로그. 외부 규정 인용의 미해결 연결도 별도 기록."]}
    write_json_atomic(EXPORT_DIR / "handoff_summary.json", HANDOFF_SUMMARY)
    print("JSONL 저장 완료:", EXPORT_DIR)
    show_table([{key: d[key] for key in ["doc_id", "title", "source_type", "extraction_status"]} for d in EXPORT_DOCUMENTS],
               ["doc_id", "title", "source_type", "extraction_status"], limit=10)


JSONL 저장 완료: /Users/gohanjun/Desktop/SNU/빅데이터핀테크응용사례특강/이화여대_학사행정_RAG/10_학사선별/학부_학사_통합_v4_전달_full


doc_id,title,source_type,extraction_status
0018f87b395a08efc93bef06,[학부] 2021학년도 2학기 선배라면 장학금 신청 안내(~10/22),공지,extracted_draft
0019b2e55ba0e253ea985a51,붙임1._2025학년도_2학기_고졸_후학습자_장학사업_업무처리기준.pdf,안내,extracted_draft
001e2f199ded0fe0dfe2419a,"[장학] 2025-1학기 경원문화재단 장학금 지원자 선발 안내(~3/28 금, 16:00까지로 연장)",공지,extracted_draft
002c274f5aa8c72629dcbb18,[학부] 2026학년도 1학기 본부기탁 장학금 신청 안내,공지,extracted_draft
003363dcc81502e2b12d6aa9,[장학] [학부] 2026학년도 제1학기 전공리더십 장학금 신청 안내,공지,extracted_draft
003afe0bbbfa42e30df7b4fd,[장학] 2025-2 장학금 감사편지 제출 안내 (~12/22),공지,extracted_draft
0040d77052ddba42b2996760,2026학년도_여름_계절학기_예고_교과목Courses_available_in_the_Summer_Session_2026 (1).pdf,공지,extracted_draft
004ee72103f54f9aac0a89f1,2009-1 이화복지장학금 추가신청 안내,공지,extracted_draft
0054cd727e229fa9e61627f2,[장학] [학부] 2026 상반기 AI·ICT 인재 장학생 선발 안내 - 이화여자대학교 조형예술대학,공지,extracted_draft
006ca15650308f2150af2111,[학부] 2026학년도 1학기 학부 전공리더십 장학금 신청 안내,공지,extracted_draft


## 16. BGE-M3 임베딩·Qdrant 적재와 명세

In [22]:
PAYLOAD_FIELDS = {
    "chunk_id": "string", "doc_id": "string", "text": "string", "section": "string", "source_url": "string", "title": "string",
    "source_type": "규정|안내|FAQ|공지|교육과정", "applicability.department.status": "specified|all|unknown",
    "applicability.department.values": "string[]", "applicability.admission_year.status": "specified|all|unknown",
    "applicability.admission_year.from": "integer|null", "applicability.admission_year.to": "integer|null",
    "applicability.student_status.status": "specified|all|unknown", "applicability.student_status.values": "string[]",
    "applicability.basis": "string", "dates.posted_at": "date|null", "dates.amended_at": "date|null",
    "dates.effective_from": "date|null", "dates.effective_until": "date|null", "dates.target_term": "YYYY-semester|null",
    "related_chunk_ids": "string[]", "metadata_review_required": "boolean", "quality_flags": "string[]",
    "ingestion_status": "ready", "ingestion_reasons": "string[]",
    "parent_doc_id": "string|null", "extraction_status": "extracted_draft|partial",
}
QDRANT_SPEC = None


def handoff_load_qdrant():
    import importlib.metadata
    import numpy as np
    from FlagEmbedding import BGEM3FlagModel
    from huggingface_hub import HfApi, snapshot_download
    from qdrant_client import QdrantClient, models
    # 실제 모델 커밋을 고정해서 저장한다. 비밀키는 환경변수에서만 읽는다.
    resolved_revision = HfApi().model_info(EMBEDDING_MODEL, revision=EMBEDDING_REVISION).sha
    model_path = snapshot_download(repo_id=EMBEDDING_MODEL, revision=resolved_revision)
    embedding_model = BGEM3FlagModel(model_path, use_fp16=False, devices=EMBEDDING_DEVICE)
    tokenizer = embedding_model.tokenizer
    eligible = []; deferred = []
    documents_by_id = {d["doc_id"]: d for d in EXPORT_DOCUMENTS}
    for chunk in EXPORT_CHUNKS:
        reasons = handoff_ingestion_reasons(chunk, documents_by_id.get(chunk["doc_id"]))
        if chunk.get("ingestion_status") != "ready": reasons.append("not_marked_ready")
        if reasons:
            deferred.append({"chunk_id": chunk["chunk_id"], "reason": "quality_gate", "details": sorted(set(reasons))})
            continue
        length = len(tokenizer.encode(chunk["text"], add_special_tokens=True, truncation=False))
        if length > EMBEDDING_MAX_TOKENS:
            deferred.append({"chunk_id": chunk["chunk_id"], "reason": "embedding_token_limit_without_truncation", "tokens": length})
        else:
            eligible.append(chunk)
    handoff_write_jsonl(EXPORT_DIR / "embedding_deferred.jsonl", deferred)
    if not eligible: raise ValueError("토큰 상한 내의 적재 가능한 청크가 없습니다.")
    vector_settings = json.dumps({"dense": QDRANT_DENSE_NAME, "sparse": QDRANT_SPARSE_NAME, "sparse_enabled": QDRANT_USE_SPARSE}, sort_keys=True)
    identity = hashlib.sha256((resolved_revision + vector_settings + HANDOFF_SUMMARY["files"]["chunks.jsonl"]["sha256"]).encode()).hexdigest()[:12]
    collection_name = f"{QDRANT_COLLECTION_PREFIX}_{HANDOFF_MODE}_{identity}"
    if QDRANT_MODE == "local":
        client = QdrantClient(path=str(PROJECT_ROOT / "11_Qdrant" / HANDOFF_MODE))
    elif QDRANT_MODE == "server":
        url = os.environ.get("QDRANT_URL")
        if not url: raise ValueError("server 모드에는 QDRANT_URL 환경변수가 필요합니다.")
        client = QdrantClient(url=url, api_key=os.environ.get("QDRANT_API_KEY"))
    else:
        raise ValueError("QDRANT_MODE는 local 또는 server이어야 합니다.")
    try:
        if not client.collection_exists(collection_name):
            client.create_collection(collection_name=collection_name,
                vectors_config={QDRANT_DENSE_NAME: models.VectorParams(size=1024, distance=models.Distance.COSINE)},
                sparse_vectors_config={QDRANT_SPARSE_NAME: models.SparseVectorParams()} if QDRANT_USE_SPARSE else None)
        info = client.get_collection(collection_name)
        dense = info.config.params.vectors
        if not isinstance(dense, dict) or QDRANT_DENSE_NAME not in dense or dense[QDRANT_DENSE_NAME].size != 1024:
            raise ValueError("기존 컬렉션의 dense 구조가 다릅니다.")
        if dense[QDRANT_DENSE_NAME].distance != models.Distance.COSINE: raise ValueError("기존 컬렉션의 거리 방식이 다릅니다.")
        if QDRANT_USE_SPARSE and QDRANT_SPARSE_NAME not in (info.config.params.sparse_vectors or {}):
            raise ValueError("기존 컬렉션의 sparse 구조가 다릅니다.")
        indexed_fields = {"ingestion_status": models.PayloadSchemaType.KEYWORD, "doc_id": models.PayloadSchemaType.KEYWORD, "source_type": models.PayloadSchemaType.KEYWORD,
            "applicability.department.status": models.PayloadSchemaType.KEYWORD, "applicability.department.values": models.PayloadSchemaType.KEYWORD,
            "applicability.admission_year.status": models.PayloadSchemaType.KEYWORD, "applicability.admission_year.from": models.PayloadSchemaType.INTEGER,
            "applicability.admission_year.to": models.PayloadSchemaType.INTEGER, "applicability.student_status.status": models.PayloadSchemaType.KEYWORD,
            "applicability.student_status.values": models.PayloadSchemaType.KEYWORD}
        if QDRANT_MODE == "server":
            for field, schema in indexed_fields.items():
                client.create_payload_index(collection_name=collection_name, field_name=field, field_schema=schema)
        mapping = []
        for start in range(0, len(eligible), EMBEDDING_BATCH_SIZE):
            batch = eligible[start:start+EMBEDDING_BATCH_SIZE]
            encoded = embedding_model.encode([c["text"] for c in batch], batch_size=EMBEDDING_BATCH_SIZE,
                max_length=EMBEDDING_MAX_TOKENS, return_dense=True, return_sparse=QDRANT_USE_SPARSE, return_colbert_vecs=False)
            vectors = np.asarray(encoded["dense_vecs"])
            if vectors.shape != (len(batch), 1024) or not np.isfinite(vectors).all(): raise ValueError("dense 벡터 차원/값 오류")
            points = []
            for index, chunk in enumerate(batch):
                point_id = str(uuid.uuid5(uuid.NAMESPACE_URL, collection_name + ":" + chunk["chunk_id"]))
                vector = {QDRANT_DENSE_NAME: vectors[index].tolist()}
                if QDRANT_USE_SPARSE:
                    sparse = sorted((int(k), float(v)) for k, v in encoded["lexical_weights"][index].items() if float(v) != 0)
                    vector[QDRANT_SPARSE_NAME] = models.SparseVector(indices=[k for k, _ in sparse], values=[v for _, v in sparse])
                points.append(models.PointStruct(id=point_id, vector=vector, payload=chunk))
                mapping.append({"point_id": point_id, "chunk_id": chunk["chunk_id"], "doc_id": chunk["doc_id"]})
            client.upsert(collection_name=collection_name, points=points, wait=True)
            print(f"Qdrant 적재 {min(start+len(batch),len(eligible)):,}/{len(eligible):,}", end="\r")
        actual = client.count(collection_name=collection_name, exact=True).count
        if actual != len(eligible): raise ValueError("적재된 point 건수가 인계 청크와 다릅니다.")
        handoff_write_jsonl(EXPORT_DIR / "qdrant_point_ids.jsonl", mapping)
        return {
            "status": "loaded_with_deferred_chunks" if deferred else "loaded", "collection_name": collection_name,
            "mode": QDRANT_MODE, "local_path": str(PROJECT_ROOT / "11_Qdrant" / HANDOFF_MODE) if QDRANT_MODE == "local" else None,
            "connection": "QDRANT_URL and QDRANT_API_KEY environment variables" if QDRANT_MODE == "server" else "local_path; shared server requires server mode",
            "payload_fields": PAYLOAD_FIELDS, "payload_schema_file": "schema.json#chunks",
            "embedding": {"model": EMBEDDING_MODEL, "revision": resolved_revision, "requested_revision": EMBEDDING_REVISION,
                "library": "FlagEmbedding", "library_version": importlib.metadata.version("FlagEmbedding"),
                "device": EMBEDDING_DEVICE, "max_tokens": EMBEDDING_MAX_TOKENS, "truncation": False},
            "dense": {"name": QDRANT_DENSE_NAME, "dimension": 1024, "distance": "Cosine"},
            "sparse": {"enabled": QDRANT_USE_SPARSE, "name": QDRANT_SPARSE_NAME if QDRANT_USE_SPARSE else None,
                       "encoder": "BGE-M3 lexical weights", "distance": "Dot" if QDRANT_USE_SPARSE else None},
            "qdrant_client_version": importlib.metadata.version("qdrant-client"), "indexed_fields": list(indexed_fields) if QDRANT_MODE == "server" else [],
            "point_count": actual, "deferred_count": len(deferred), "chunks_sha256": HANDOFF_SUMMARY["files"]["chunks.jsonl"]["sha256"],
            "query_example": "query_example.py", "secrets_in_artifact": False,
        }
    finally:
        client.close()


if HANDOFF_READY:
    if RUN_QDRANT_LOAD:
        try:
            QDRANT_SPEC = handoff_load_qdrant()
        except (ImportError, ModuleNotFoundError) as error:
            raise ImportError("DB 적재에는 FlagEmbedding, qdrant-client, huggingface-hub가 필요합니다. 설정 안내의 설치 명령을 현재 커널에서 실행하세요.") from error
    else:
        QDRANT_SPEC = {"status": "not_loaded", "collection_name": None, "planned_collection_prefix": QDRANT_COLLECTION_PREFIX,
            "mode": QDRANT_MODE, "payload_fields": PAYLOAD_FIELDS, "payload_schema_file": "schema.json#chunks",
            "embedding": {"model": EMBEDDING_MODEL, "revision": None, "requested_revision": EMBEDDING_REVISION},
            "dense": {"name": QDRANT_DENSE_NAME, "dimension": 1024, "distance": "Cosine"},
            "sparse": {"enabled": QDRANT_USE_SPARSE, "name": QDRANT_SPARSE_NAME if QDRANT_USE_SPARSE else None, "encoder": "BGE-M3 lexical weights"},
            "point_count": 0, "query_example": "query_example.py", "secrets_in_artifact": False}
        print("JSONL 작성 완료. 실제 모델 다운로드/DB 적재는 RUN_QDRANT_LOAD=True일 때 실행합니다.")
    write_json_atomic(EXPORT_DIR / "qdrant_spec.json", QDRANT_SPEC)
    HANDOFF_SUMMARY["embedding_and_db_status"] = QDRANT_SPEC["status"]
    write_json_atomic(EXPORT_DIR / "handoff_summary.json", HANDOFF_SUMMARY)


JSONL 작성 완료. 실제 모델 다운로드/DB 적재는 RUN_QDRANT_LOAD=True일 때 실행합니다.


## 17. 독립 조회 예시와 인계 안내

In [23]:
QUERY_EXAMPLE = r'''"""qdrant_spec.json과 같은 폴더에서 실행. 비밀키는 환경변수로만 전달."""
import json, os
from pathlib import Path
from FlagEmbedding import BGEM3FlagModel
from huggingface_hub import snapshot_download
from qdrant_client import QdrantClient, models

HERE = Path(__file__).resolve().parent
spec = json.loads((HERE / "qdrant_spec.json").read_text(encoding="utf-8"))
if not spec["status"].startswith("loaded"):
    raise RuntimeError("통합 노트북에서 RUN_QDRANT_LOAD=True로 적재를 먼저 완료하세요.")
model_path = snapshot_download(repo_id=spec["embedding"]["model"], revision=spec["embedding"]["revision"])
model = BGEM3FlagModel(model_path, use_fp16=False, devices="cpu")
client = QdrantClient(path=spec["local_path"]) if spec["mode"] == "local" else QdrantClient(
    url=os.environ["QDRANT_URL"], api_key=os.environ.get("QDRANT_API_KEY"))


def query_academic(question, limit=5, scope_filter=None):
    # 실제 조건을 아는 경우 scope_filter로 specified 범위를 적용한다.
    # unknown은 all로 간주하지 않는다. 아래 기본 예시는 세 조건 모두 제한 없음이 확인된 자료만 검색 (결과가 없을 수 있음).
    if scope_filter is None:
        scope_filter = models.Filter(must=[models.FieldCondition(
            key=f"applicability.{field}.status", match=models.MatchValue(value="all"))
            for field in ["department", "admission_year", "student_status"]])
    scope_filter = models.Filter(must=[scope_filter, models.FieldCondition(
        key="ingestion_status", match=models.MatchValue(value="ready"))])
    encoded = model.encode([question], return_dense=True, return_sparse=spec["sparse"]["enabled"], return_colbert_vecs=False)
    dense = encoded["dense_vecs"][0].tolist()
    if spec["sparse"]["enabled"]:
        weights = sorted((int(k), float(v)) for k, v in encoded["lexical_weights"][0].items() if float(v) != 0)
        sparse = models.SparseVector(indices=[k for k, _ in weights], values=[v for _, v in weights])
        response = client.query_points(collection_name=spec["collection_name"],
            prefetch=[models.Prefetch(query=dense, using=spec["dense"]["name"], filter=scope_filter, limit=20),
                      models.Prefetch(query=sparse, using=spec["sparse"]["name"], filter=scope_filter, limit=20)],
            query=models.FusionQuery(fusion=models.Fusion.RRF), query_filter=scope_filter, limit=limit, with_payload=True)
    else:
        response = client.query_points(collection_name=spec["collection_name"], query=dense, using=spec["dense"]["name"],
            query_filter=scope_filter, limit=limit, with_payload=True)
    return [{"score": p.score, "payload": p.payload} for p in response.points]


if __name__ == "__main__":
    try:
        print(json.dumps(query_academic("휴학 신청 방법이 뭐야?"), ensure_ascii=False, indent=2))
    finally:
        client.close()
'''

if HANDOFF_READY:
    (EXPORT_DIR / "query_example.py").write_text(QUERY_EXAMPLE, encoding="utf-8")
    note = f'''# 학사 자료 인계

- 문서 단위: `documents.jsonl` ({len(EXPORT_DOCUMENTS):,}개 문서, 문서당 한 행)
- 청크 단위: `chunks.jsonl` ({len(EXPORT_CHUNKS):,}개 청크)
- 스키마: `schema.json`
- Qdrant 명세: `qdrant_spec.json` (현재 상태: {QDRANT_SPEC['status']})
- 조회 예시: `query_example.py`
- 실패 기록: `extraction_failures.jsonl` (수집 전체의 PDF/HWP류 추출 실패; 정상 문서로 적재하지 않음)
- 미해결 인용: `unresolved_references.jsonl`
- 표본 밖 부모 문서 참조: `referenced_documents.jsonl`
- 본문 결함 보류: `quarantined_documents.jsonl` / `quarantined_chunks.jsonl` (DB 적재 대상 아님)
- 통과/보류 기준 및 전체 재조립 문서 집계: `quality_gate.json`

학과/입학 연도/학생 상태는 specified, all, unknown을 구분합니다. all은 제한 없음의 원문 근거가 있을 때만 사용합니다. 원문에 없는 날짜는 null이며 게시일을 시행일로 쓰지 않습니다. 날짜 근거/복수 후보, 과거 학기와 부칙은 청크에 남습니다.

unknown/null은 그대로 허용합니다. 일부 본문 누락·문자 손상·FAQ 연결 미확정은 정상 파일에서 제외하고 보류 파일에 남깁니다.

메타데이터는 규칙 기반 초안입니다. 여러 조건이 섞인 표/문장은 unknown과 검토 메모를 유지합니다. PDF/HWP의 기존 파싱 플래그도 보존하므로 표·레이아웃 의미 검토가 필요한 자료가 있습니다.

실제 DB 적재는 RUN_QDRANT_LOAD=True일 때 동작합니다. not_loaded 명세는 계획이며 적재 완료 증빙이 아닙니다. 토큰 상한을 넘는 FAQ/긴 문장은 잘라 임베딩하지 않고 embedding_deferred.jsonl에 보류합니다. 적재 후에는 모델 커밋·라이브러리 버전·벡터 이름과 차원·거리·건수를 실제 값으로 기록합니다.

local 모드는 노트북 PC의 로컬 DB입니다. 현선님이 다른 PC에서 접속하는 공유 DB는 server 모드로 적재하고 접속 주소/키를 별도로 전달하세요. 비밀키는 Git/JSONL/명세에 넣지 않습니다.
'''
    (EXPORT_DIR / "인계_안내.md").write_text(note, encoding="utf-8")
    print("통합 처리 완료:", EXPORT_DIR)
    print("전달 파일: documents.jsonl / chunks.jsonl / schema.json / qdrant_spec.json / query_example.py")
    print("실제 DB 상태:", QDRANT_SPEC["status"])


통합 처리 완료: /Users/gohanjun/Desktop/SNU/빅데이터핀테크응용사례특강/이화여대_학사행정_RAG/10_학사선별/학부_학사_통합_v4_전달_full
전달 파일: documents.jsonl / chunks.jsonl / schema.json / qdrant_spec.json / query_example.py
실제 DB 상태: not_loaded


## 확대 실행

처음에는 전달 표본 25개를 확인하세요. `HANDOFF_MODE="full"`로 바꾸면 전체 통과 문서만 인계 파일에 저장합니다. 학사 선별의 전체 원본 스캔은 기본 설정에서도 수행됩니다.

보류 자료는 사유를 확인해 원본 추출이나 문서 범위를 보완한 뒤 재실행하세요. `unknown`을 `all`로 바꾸거나 게시일을 시행일로 대신 쓰지 않습니다. 코드·선별 설정이 바뀌면 `RUN_NAME`도 변경해 이전 체크포인트와 구분하세요.

## 18. 보완 사항과 인계 결과 자동 확인

실제 출력과 기존 문제 사례를 확인합니다. 조건 미확인과 본문 결함은 서로 다른 기준으로 처리합니다. 이 검사는 전체 문서의 의미 정확도 평가나 실제 DB 적재 완료를 대신하지 않습니다.

In [24]:

if HANDOFF_READY:
    checks = []
    def verify(name, passed, evidence):
        checks.append({"check": name, "passed": bool(passed), "evidence": evidence})
    known_journals = {"118e5b80d34f13df7967cc51", "dc7eea3b807863fb7c16cf95", "e30faf255aa4750e009d759a", "2add2989bae8b3bb1b588f5c"}
    verify("학술지 규정 오분류 차단", not (known_journals & HANDOFF_INGESTION_READY.keys()), "기존 문제 문서 4개를 전체 통과 집합과 비교")
    certificate_id = "0118832698479d858a5e356a"
    certificate = HANDOFF_INGESTION_READY.get(certificate_id)
    steps = ["1. 유레카통합행정 로그인", "2. 유레카통합행정 로그인 후", "3. 플러그인 설치 후", "5. 증명서 목록 확인 후"]
    verify("짧은 증명서 신청 절차 보존", certificate is not None and all(step in certificate["text"] for step in steps), "기존 누락 단계 1·2·3·5를 재조립 본문과 비교")
    verify("문서 20~30개 표본", HANDOFF_MODE == "full" or 20 <= len(EXPORT_DOCUMENTS) <= 30, len(EXPORT_DOCUMENTS))
    verify("본문 결함 인계/적재 차단", all(c.get("ingestion_status") == "ready" and not handoff_ingestion_reasons(c, HANDOFF_INGESTION_READY.get(c["doc_id"])) for c in EXPORT_CHUNKS), len(EXPORT_CHUNKS))
    verify("부분 본문 차단 기준", bool(handoff_ingestion_reasons({"text": "신청 안내", "extraction_status": "partial"})), "partial 합성 사례")
    verify("일반 미확인 메타데이터 허용", not handoff_ingestion_reasons({"text": "휴학 신청 안내", "metadata_review_required": True}), "unknown/null 검토 표시 자체로 보류하지 않음")
    faq_chunks = [c for c in EXPORT_CHUNKS if c["source_type"] == "FAQ"]
    verify("FAQ 질문·답변 함께 보존", bool(faq_chunks) and all(c["unit_type"] == "faq" and "질문:" in c["text"] and "답변:" in c["text"] for c in faq_chunks), len(faq_chunks))
    explicit = handoff_applicability("", "25학번(2025학년도) 전공별 교과과정 안내", {"title": "전공별 교과과정"})
    verify("원문 근거의 학번 해석", explicit["admission_year"] == {"status": "specified", "from": 2025, "to": 2025}, explicit["admission_year"])
    verify("기존 규정 필드·ID·연결 검사", HANDOFF_CHECK["ids_and_required_fields"] == "passed", HANDOFF_CHECK)
    report = {"version": HANDOFF_VERSION, "checked_at": datetime.now(ZoneInfo("Asia/Seoul")).isoformat(timespec="seconds"),
              "checks": checks, "passed": all(c["passed"] for c in checks), "db_status": QDRANT_SPEC["status"],
              "limitations": ["기존 문제 사례 및 구조 검증; 전체 의미 오분류율을 측정한 결과가 아님", "실제 임베딩·Qdrant 적재는 RUN_QDRANT_LOAD=True로 별도 실행"]}
    write_json_atomic(EXPORT_DIR / "verification.json", report)
    show_table(checks, ["check", "passed", "evidence"], limit=12)
    if not report["passed"]: raise AssertionError("보완 검사 실패: verification.json과 관련 원본을 확인하세요.")
    print("보완 검사 통과 / 문서", len(EXPORT_DOCUMENTS), "/ 청크", len(EXPORT_CHUNKS), "/ DB", QDRANT_SPEC["status"])


check,passed,evidence
학술지 규정 오분류 차단,True,기존 문제 문서 4개를 전체 통과 집합과 비교
짧은 증명서 신청 절차 보존,True,기존 누락 단계 1·2·3·5를 재조립 본문과 비교
문서 20~30개 표본,True,7163
본문 결함 인계/적재 차단,True,16863
부분 본문 차단 기준,True,partial 합성 사례
일반 미확인 메타데이터 허용,True,unknown/null 검토 표시 자체로 보류하지 않음
FAQ 질문·답변 함께 보존,True,40
원문 근거의 학번 해석,True,"{'status': 'specified', 'from': 2025, 'to': 2025}"
기존 규정 필드·ID·연결 검사,True,"{'json_schema': 'required_fields_and_links_only; optional jsonschema not installed', 'documents': 7163, 'chunks': 16863, 'ids_and_required_fields': 'passed'}"


보완 검사 통과 / 문서 7163 / 청크 16863 / DB not_loaded
